In [ ]:
import os
import json
import zipfile
import random
from copy import deepcopy
from datetime import datetime, UTC
from functools import partial
from pathlib import Path
from typing import Dict, List, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SEED = 42
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "sentence-transformers/all-roberta-large-v1"
MAX_LEN = 256

STAGE1_BS = 2
STAGE2_BS = 4
PRED_BS   = 32
ENC_BS    = 16
VAL_RATIO = 0.10

USE_SYNTHETIC_STAGE1 = True
USE_ASPECTS = True

ASPECTS_TO_USE = "action_outcome"
FUSION_TYPE = "gated"
FINAL_RETRAIN_ON_FULL_DEV = True

SYNTH_PATH         = "/kaggle/input/datasets/anisoaraana/similarity/synthetic_data_for_classification.jsonl"
DEV_TRACK_A_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl"
TEST_TRACK_A_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl"
TEST_TRACK_B_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl"

TEST_TRACK_A_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a_labels.jsonl"
TEST_TRACK_B_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl"

ASPECTS_CACHE_PATH = "/kaggle/input/datasets/anisoaraana/similarity/clean_extracted_aspects.json"

OUT_DIR = "/kaggle/working/"
OUT_TRACK_A = OUT_DIR + "track_a.jsonl"
OUT_TRACK_B = OUT_DIR + "track_b.npy"

RUNS_DIR    = OUT_DIR + "runs"
RESULTS_LOG_PATH = OUT_DIR + "experiment_results.jsonl"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

    try:
        if hasattr(torch.backends.cuda, "enable_flash_sdp"):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
    except Exception:
        pass

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def make_run_name() -> str:
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    mode = "aspects" if USE_ASPECTS else "baseline"
    synth = "synth" if USE_SYNTHETIC_STAGE1 else "nosynth"
    return f"{timestamp}_{mode}_{ASPECTS_TO_USE}_{FUSION_TYPE}_{synth}_seed{SEED}"

def save_json(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def append_jsonl(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path: str, model: nn.Module, epoch: int, metrics: Dict[str, Any], config: Dict[str, Any]) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "metrics": metrics,
            "config": config,
        },
        path,
    )

def load_checkpoint(path: str, model: nn.Module) -> Dict[str, Any]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

def compute_binary_metrics(preds, golds):
    n = len(golds)
    if n == 0:
        return {"accuracy": 0.0, "macro_f1": 0.0, "tp": 0, "tn": 0, "fp": 0, "fn": 0}

    correct = sum(p == g for p, g in zip(preds, golds))
    accuracy = correct / n

    tp = sum(p and g for p, g in zip(preds, golds))
    tn = sum((not p) and (not g) for p, g in zip(preds, golds))
    fp = sum(p and (not g) for p, g in zip(preds, golds))
    fn = sum((not p) and g for p, g in zip(preds, golds))

    prec_a = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_a = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_a = 2 * prec_a * rec_a / (prec_a + rec_a) if (prec_a + rec_a) > 0 else 0.0

    prec_b = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_b = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_b = 2 * prec_b * rec_b / (prec_b + rec_b) if (prec_b + rec_b) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "macro_f1": (f1_a + f1_b) / 2,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

def get_aspect_info(aspects_to_use):
    if aspects_to_use == "all":
        return ["theme", "action", "outcome"], 3
    elif aspects_to_use == "theme":
        return ["theme"], 1
    elif aspects_to_use == "action":
        return ["action"], 1
    elif aspects_to_use == "outcome":
        return ["outcome"], 1
    elif aspects_to_use == "theme_action":
        return ["theme", "action"], 2
    elif aspects_to_use == "theme_outcome":
        return ["theme", "outcome"], 2
    elif aspects_to_use == "action_outcome":
        return ["action", "outcome"], 2
    else:
        return ["theme", "action", "outcome"], 3

ASPECT_NAMES, ASPECT_MULTIPLIER = get_aspect_info(ASPECTS_TO_USE)
print(f"Using aspects: {ASPECTS_TO_USE} -> {ASPECT_NAMES}")
print(f"Fusion type: {FUSION_TYPE}")

class PreExtractedAspectLoader:
    def __init__(self, cache_path: str):
        self.cache_path = cache_path
        self.aspects_cache = {}
        self.lookup_hits = 0
        self.lookup_misses = 0
        self._load_cache()
        print(f"✓ PreExtractedAspectLoader initialized with {len(self.aspects_cache)} entries")

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r", encoding="utf-8") as f:
                self.aspects_cache = json.load(f)
            print(f"  Loaded {len(self.aspects_cache)} pre-extracted aspect entries")
        else:
            print(f"  WARNING: Cache file not found at {self.cache_path}")
            print("  Will use empty aspects for all stories")

    def _normalize_text(self, text: str) -> str:
        return " ".join((text or "").split())

    def extract_aspects(self, text: str, story_id: Optional[str] = None) -> Dict[str, str]:
        norm_text = self._normalize_text(text)
        if norm_text in self.aspects_cache:
            self.lookup_hits += 1
            cached = self.aspects_cache[norm_text]
            return {
                "course_of_action": cached.get("coa", ""),
                "outcomes": cached.get("outcomes", ""),
                "abstract_theme": cached.get("theme", ""),
            }

        self.lookup_misses += 1
        return {
            "course_of_action": "",
            "outcomes": "",
            "abstract_theme": "",
        }

    def get_stats(self) -> Dict[str, Any]:
        total = self.lookup_hits + self.lookup_misses
        return {
            "hits": self.lookup_hits,
            "misses": self.lookup_misses,
            "total": total,
            "hit_rate": (self.lookup_hits / total) if total > 0 else 0.0,
        }

class AspectAwareEncoderWithFusion(nn.Module):
    def __init__(self, base_model, hidden_dim=None, aspect_dim=128,
                 aspect_names=None, fusion_type="attention", initial_aspect_weight=0.30):
        super().__init__()
        self.base_model = base_model
        self.hidden_dim = hidden_dim or base_model.config.hidden_size
        self.aspect_dim = aspect_dim
        self.aspect_names = aspect_names or ["theme", "action", "outcome"]
        self.fusion_type = fusion_type
        self.num_aspects = len(self.aspect_names)
        self.aspect_embedding_cache = {}

        if self.fusion_type == "attention":
            self.num_heads = min(4, self.num_aspects)
            self.aspect_dim = ((aspect_dim + self.num_heads - 1) // self.num_heads) * self.num_heads
            print(f"  Adjusted aspect_dim to {self.aspect_dim} for divisibility by {self.num_heads}")

        init_weight = torch.tensor(initial_aspect_weight, dtype=torch.float32).clamp(1e-4, 1 - 1e-4)
        self.aspect_weight_raw = nn.Parameter(torch.logit(init_weight))

        self.aspect_encoders = nn.ModuleDict()
        for aspect_name in self.aspect_names:
            self.aspect_encoders[aspect_name] = nn.Sequential(
                nn.Linear(self.hidden_dim, self.aspect_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.aspect_dim, self.aspect_dim)
            )

        self.aspect_text_encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.aspect_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        for param in self.aspect_text_encoder.parameters():
            param.requires_grad = False
        self.aspect_text_encoder.eval()

        if fusion_type == "attention":
            self.aspect_attention = nn.MultiheadAttention(
                embed_dim=self.aspect_dim,
                num_heads=self.num_heads,
                batch_first=True,
                dropout=0.1
            )
            self.aspect_query = nn.Parameter(torch.randn(1, 1, self.aspect_dim))
        elif fusion_type == "gated":
            self.gates = nn.Sequential(
                nn.Linear(self.aspect_dim * self.num_aspects, self.num_aspects),
                nn.Sigmoid()
            )
        elif fusion_type == "concat":
            self.concat_proj = nn.Linear(self.num_aspects * self.aspect_dim, self.aspect_dim)

        self.final_proj = nn.Sequential(
            nn.Linear(self.aspect_dim, self.hidden_dim),
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(0.1)
        )

        self.base_model.config.use_cache = False
        self.aspect_text_encoder.config.use_cache = False

        print(" AspectAwareEncoderWithFusion initialized")
        print(f"  - Aspects: {self.aspect_names}")
        print(f"  - Num aspects: {self.num_aspects}")
        print(f"  - Aspect dim: {self.aspect_dim}")
        print(f"  - Fusion type: {fusion_type}")
        print(f"  - Learnable aspect weight: {self.get_aspect_weight():.3f}")

    def encode_aspect_text(self, text: str) -> torch.Tensor:
        if not text or len(text) < 5:
            return torch.zeros(self.hidden_dim, device=next(self.base_model.parameters()).device)

        cache_key = " ".join(text.split())
        if cache_key in self.aspect_embedding_cache:
            return self.aspect_embedding_cache[cache_key].to(next(self.base_model.parameters()).device)

        inputs = self.aspect_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(next(self.base_model.parameters()).device)

        with torch.no_grad():
            outputs = self.aspect_text_encoder(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)

        self.aspect_embedding_cache[cache_key] = emb.detach().cpu()
        return emb

    def _get_aspect_text(self, aspects: Dict[str, str], aspect_name: str) -> str:
        if aspect_name == "theme":
            return aspects.get("abstract_theme", "")
        elif aspect_name == "action":
            return aspects.get("course_of_action", "")
        elif aspect_name == "outcome":
            return aspects.get("outcomes", "")
        return ""

    def _encode_single_story_aspects(self, aspects: Dict[str, str]) -> torch.Tensor:
        aspect_embeddings = []
        device = next(self.base_model.parameters()).device

        for aspect_name in self.aspect_names:
            aspect_text = self._get_aspect_text(aspects, aspect_name)
            if aspect_text:
                text_emb = self.encode_aspect_text(aspect_text)
                aspect_emb = self.aspect_encoders[aspect_name](text_emb.unsqueeze(0)).squeeze(0)
            else:
                aspect_emb = torch.zeros(self.aspect_dim, device=device)
            aspect_embeddings.append(aspect_emb)

        return torch.stack(aspect_embeddings)

    def fuse_aspects(self, aspect_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size, num_aspects, _ = aspect_embeddings.shape

        if self.fusion_type == "attention":
            query = self.aspect_query.expand(batch_size, -1, -1)
            attended, attn_weights = self.aspect_attention(query, aspect_embeddings, aspect_embeddings)
            fused = attended.squeeze(1)
            if not self.training:
                self.last_attn_weights = attn_weights.detach().cpu()
        elif self.fusion_type == "gated":
            flat = aspect_embeddings.reshape(batch_size, -1)
            gate_weights = self.gates(flat).unsqueeze(-1)
            fused = (aspect_embeddings * gate_weights).sum(dim=1)
        else:
            fused = aspect_embeddings.reshape(batch_size, -1)
            fused = self.concat_proj(fused)

        fused = self.final_proj(fused)
        fused = F.normalize(fused, p=2, dim=1)
        return fused

    def forward(self, input_ids, attention_mask, aspects: Optional[List[Dict[str, str]]] = None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        base_emb = mean_pool(outputs, attention_mask)

        if aspects is None:
            return base_emb

        if isinstance(aspects, list):
            all_aspect_embs = [self._encode_single_story_aspects(asp) for asp in aspects]
            aspect_embeddings = torch.stack(all_aspect_embs)
            aspect_contribution = self.fuse_aspects(aspect_embeddings)
        else:
            aspect_embs = self._encode_single_story_aspects(aspects)
            aspect_contribution = self.fuse_aspects(aspect_embs.unsqueeze(0))

        aspect_weight = torch.sigmoid(self.aspect_weight_raw)
        final_emb = base_emb + aspect_weight * aspect_contribution
        final_emb = F.normalize(final_emb, p=2, dim=1)
        return final_emb

    def get_aspect_weight(self):
        return torch.sigmoid(self.aspect_weight_raw).item()

    def get_attention_weights(self):
        if hasattr(self, "last_attn_weights"):
            return self.last_attn_weights
        return None

class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                anchor = obj.get("anchor_text", "")
                ta = obj.get("text_a", "")
                tb = obj.get("text_b", "")
                closer = obj.get("text_a_is_closer")
                if not anchor or not ta or not tb:
                    continue
                if len(anchor) < 20 or len(ta) < 20 or len(tb) < 20:
                    continue
                pos, neg = (ta, tb) if closer else (tb, ta)
                self.data.append((anchor.strip(), pos.strip(), neg.strip()))
        print(f"SyntheticTripletDataset: {len(self.data)} triplets")

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class AspectAwareTrackATrainDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "label": bool(obj.get("text_a_is_closer", False)),
                    "anchor_id": f"anchor_{idx}",
                    "a_id": f"a_{idx}",
                    "b_id": f"b_{idx}",
                })
        print(f"AspectAwareTrackATrainDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "label": torch.tensor(item["label"], dtype=torch.float32),
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackATestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "anchor_id": f"test_anchor_{idx}",
                    "a_id": f"test_a_{idx}",
                    "b_id": f"test_b_{idx}",
                })
        print(f"AspectAwareTrackATestDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackBTestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.texts = []
        self.ids = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.texts.append(obj["text"])
                self.ids.append(f"test_b_{idx}")
        print(f"AspectAwareTrackBTestDataset: {len(self.texts)} texts")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        return {
            "text": text,
            "aspects": self.aspect_loader.extract_aspects(text, self.ids[idx]),
            "id": self.ids[idx],
        }

def tokenize_texts(tokenizer, texts, max_len):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

def aspect_aware_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "label": torch.stack([b["label"] for b in batch]),
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_test_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_b_collate(batch, tokenizer, max_len):
    enc = tokenize_texts(tokenizer, [b["text"] for b in batch], max_len)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "aspects": [b["aspects"] for b in batch],
        "id": [b["id"] for b in batch],
    }

def stratified_split_aspect_dataset(dataset, val_ratio=VAL_RATIO, seed=SEED):
    labels = np.array([1 if item["label"] else 0 for item in dataset.data])
    indices = np.arange(len(labels))
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    train_idx, val_idx = next(splitter.split(indices, labels))
    train_ds = Subset(dataset, train_idx.tolist())
    val_ds = Subset(dataset, val_idx.tolist())
    print(f"Dataset split (stratified): train={len(train_ds)} val={len(val_ds)} (val_ratio={val_ratio:.2f})")
    return train_ds, val_ds

def evaluate_aspect_aware_loader(model, loader, criterion):
    model.eval()
    losses, preds, golds = [], [], []

    with torch.no_grad():
        for batch in loader:
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            losses.append(criterion(score, labels).item())

            preds.extend((score > 0).cpu().tolist())
            golds.extend(labels.bool().cpu().tolist())

    metrics = compute_binary_metrics(preds, golds)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    model.train()
    return metrics

def train_stage1_triplet(model, tokenizer, dataset, epochs=1):
    if len(dataset) == 0:
        print("No synthetic data — skipping Stage 1")
        return model

    loader = DataLoader(
        dataset,
        batch_size=STAGE1_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()

    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage1 Epoch {ep+1}")
        running = 0.0

        for anchors, positives, negatives in loop:
            texts = list(anchors) + list(positives) + list(negatives)
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(DEVICE)

            emb = model(enc["input_ids"], enc["attention_mask"], aspects=None)

            B = len(anchors)
            a, p, n = emb[:B], emb[B:2*B], emb[2*B:]
            loss = F.triplet_margin_loss(a, p, n, margin=0.5)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running += loss.item()
            loop.set_postfix(loss=running / (loop.n + 1))

    model.eval()
    print("Stage 1 finished.")
    return model

def train_stage2_rank_aspect_aware(model, tokenizer, train_dataset, val_dataset=None, epochs=2,
                                   run_dir=None, experiment_config=None):
    train_loader = DataLoader(
        train_dataset,
        batch_size=STAGE2_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=PRED_BS,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

    param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
    for encoder in model.aspect_encoders.values():
        param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
    param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
    param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})

    if hasattr(model, "aspect_attention"):
        param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
    if hasattr(model, "gates"):
        param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
    if hasattr(model, "concat_proj"):
        param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

    optimizer = torch.optim.AdamW(param_groups)
    criterion = nn.BCEWithLogitsLoss()
    model.train()

    weight_history = []
    epoch_history = []
    best_val_metrics = None
    best_checkpoint_path = None
    best_epoch = None

    if run_dir:
        ensure_dir(run_dir)
        best_checkpoint_path = os.path.join(run_dir, "best_model.pt")

    for ep in range(epochs):
        loop = tqdm(train_loader, desc=f"Stage2 Aspect Epoch {ep+1}")
        running_loss = 0.0

        for batch_idx, batch in enumerate(loop):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            loss = criterion(score, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=running_loss / (batch_idx + 1))

        current_weight = model.get_aspect_weight()
        weight_history.append(current_weight)
        print(f"  Epoch {ep+1} - Learnable aspect weight: {current_weight:.4f}")

        epoch_record = {
            "epoch": ep + 1,
            "train_loss": running_loss / max(len(train_loader), 1),
            "aspect_weight": current_weight,
        }

        if val_loader is not None:
            val_metrics = evaluate_aspect_aware_loader(model, val_loader, criterion)
            epoch_record["val"] = val_metrics
            print(
                f"  Epoch {ep+1} - Val loss: {val_metrics['loss']:.4f}  "
                f"Val acc: {val_metrics['accuracy']*100:.2f}%  "
                f"Val macro-F1: {val_metrics['macro_f1']*100:.2f}%"
            )
            if best_val_metrics is None or val_metrics["macro_f1"] > best_val_metrics["macro_f1"] or (
                val_metrics["macro_f1"] == best_val_metrics["macro_f1"] and val_metrics["loss"] < best_val_metrics["loss"]
            ):
                best_val_metrics = dict(val_metrics)
                best_epoch = ep + 1
                if best_checkpoint_path:
                    save_checkpoint(best_checkpoint_path, model, best_epoch, best_val_metrics, experiment_config or {})
                    print(f"  Saved new best checkpoint to {best_checkpoint_path}")

        epoch_history.append(epoch_record)

    if best_checkpoint_path and os.path.exists(best_checkpoint_path):
        checkpoint = load_checkpoint(best_checkpoint_path, model)
        best_epoch = checkpoint["epoch"]
        best_val_metrics = checkpoint["metrics"]
        print(
            f"Reloaded best aspect-aware checkpoint from epoch {best_epoch} "
            f"(val macro-F1={best_val_metrics['macro_f1']*100:.2f}%)"
        )

    if run_dir:
        save_json(
            os.path.join(run_dir, "training_history.json"),
            {
                "mode": "aspect_aware",
                "epochs": epoch_history,
                "best_epoch": best_epoch,
                "best_val_metrics": best_val_metrics,
                "weight_history": weight_history,
            },
        )

    model.eval()
    print(f"Stage 2 aspect-aware training finished. Final aspect weight: {model.get_aspect_weight():.4f}")
    print(f"Weight history: {[f'{w:.4f}' for w in weight_history]}")
    return model, best_epoch

def predict_aspect_aware_trackA(model, tokenizer, test_path, out_path, aspect_loader):
    ds = AspectAwareTrackATestDataset(test_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=PRED_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_test_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    preds = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict Track A with aspects"):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            preds.extend((sim_a > sim_b).cpu().tolist())

    with open(test_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line, p in zip(fin, preds):
            obj = json.loads(line)
            obj["text_a_is_closer"] = bool(p)
            fout.write(json.dumps(obj) + "\n")

    print("Track A predictions saved:", out_path)
    return preds

def build_trackB_embeddings_aspect_aware(model, tokenizer, in_path, out_npy, aspect_loader):
    ds = AspectAwareTrackBTestDataset(in_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=ENC_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_b_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    embs = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Encode Track B with aspects"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            emb = model(input_ids, attention_mask, batch["aspects"])
            embs.append(emb.cpu().numpy().astype(np.float32))

    X = np.vstack(embs)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-12, None)
    np.save(out_npy, X)
    print(f"Track B embeddings saved: {out_npy} shape={X.shape}")
    return X

def evaluate_trackA(predictions_path, gold_labels_path):
    preds, golds = [], []
    with open(predictions_path, encoding="utf-8") as fp, open(gold_labels_path, encoding="utf-8") as fg:
        for line_p, line_g in zip(fp, fg):
            preds.append(bool(json.loads(line_p)["text_a_is_closer"]))
            golds.append(bool(json.loads(line_g)["text_a_is_closer"]))

    metrics = compute_binary_metrics(preds, golds)
    metrics["n"] = len(golds)
    metrics["correct"] = sum(p == g for p, g in zip(preds, golds))

    print("\n" + "="*55)
    print("  TRACK A — TEST SET EVALUATION")
    print("="*55)
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def evaluate_trackB(embeddings_npy, gold_labels_path, test_track_b_path):
    X = np.load(embeddings_npy)
    text2idx = {}
    with open(test_track_b_path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            text2idx[json.loads(line)["text"].strip()] = idx

    correct = 0
    total = 0
    missing = 0
    golds, decisions = [], []

    with open(gold_labels_path, encoding="utf-8") as fg:
        for line in fg:
            obj = json.loads(line)
            anchor = obj["anchor_text"].strip()
            ta = obj["text_a"].strip()
            tb = obj["text_b"].strip()
            gold = bool(obj["text_a_is_closer"])

            idx_a = text2idx.get(anchor)
            idx_ta = text2idx.get(ta)
            idx_tb = text2idx.get(tb)

            if idx_a is None or idx_ta is None or idx_tb is None:
                missing += 1
                continue

            emb_a = X[idx_a]
            emb_ta = X[idx_ta]
            emb_tb = X[idx_tb]

            pred = float(np.dot(emb_a, emb_ta)) > float(np.dot(emb_a, emb_tb))
            golds.append(gold)
            decisions.append(pred)
            if pred == gold:
                correct += 1
            total += 1

    metrics = compute_binary_metrics(decisions, golds)
    metrics["n"] = total
    metrics["missing"] = missing
    metrics["correct"] = correct
    metrics["embedding_dim"] = int(X.shape[1])

    print("\n" + "="*55)
    print("  TRACK B — TEST SET EVALUATION")
    print("="*55)
    print(f"  Embedding dim      : {metrics['embedding_dim']}")
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Skipped (no embed) : {metrics['missing']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def print_summary(res_a, res_b):
    print("\n" + "="*55)
    print("  FINAL SUMMARY")
    print("="*55)
    print(f"  Track A accuracy  : {res_a['accuracy']*100:.2f}%")
    print(f"  Track A macro-F1  : {res_a['macro_f1']*100:.2f}%")
    print(f"  Track B accuracy  : {res_b['accuracy']*100:.2f}%")
    print(f"  Track B macro-F1  : {res_b['macro_f1']*100:.2f}%")
    print("-"*55)
    print("  SemEval baselines (Narrative Team CodaBench results):")
    print("    Track A : 64.25%")
    print("    Track B : 69.25%")
    print("="*55 + "\n")

def sanity_checks(test_a, test_b, out_a, out_b):
    n_ta = sum(1 for _ in open(test_a, encoding="utf-8"))
    n_oa = sum(1 for _ in open(out_a, encoding="utf-8"))
    assert n_ta == n_oa, f"Track A line mismatch: {n_ta} vs {n_oa}"

    n_tb = sum(1 for _ in open(test_b, encoding="utf-8"))
    X = np.load(out_b)
    assert X.shape[0] == n_tb, f"Track B row mismatch: {n_tb} vs {X.shape[0]}"
    assert np.isfinite(X).all(), "Track B has NaN/Inf"
    print(f"Sanity OK — TrackA lines={n_oa}  TrackB rows={X.shape[0]} dim={X.shape[1]}")

if __name__ == "__main__":
    print("="*60)
    print("NARRATIVE SIMILARITY WITH PRE-EXTRACTED ASPECTS & LEARNABLE RESIDUAL")
    print("="*60)

    set_seed(SEED)
    configure_determinism()
    ensure_dir(RUNS_DIR)

    run_name = make_run_name()
    run_dir = os.path.join(RUNS_DIR, run_name)
    ensure_dir(run_dir)

    experiment_config = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "seed": SEED,
        "device": DEVICE,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "use_synthetic_stage1": USE_SYNTHETIC_STAGE1,
        "use_aspects": USE_ASPECTS,
        "aspects_to_use": ASPECTS_TO_USE,
        "aspect_names": ASPECT_NAMES,
        "fusion_type": FUSION_TYPE,
        "final_retrain_on_full_dev": FINAL_RETRAIN_ON_FULL_DEV,
    }
    save_json(os.path.join(run_dir, "config.json"), experiment_config)

    print(f"SEED = {SEED}")
    print(f"RUN_NAME = {run_name}")
    print(f"ASPECTS_TO_USE = '{ASPECTS_TO_USE}'")
    print(f"FUSION_TYPE = '{FUSION_TYPE}'")
    print(f"FINAL_RETRAIN_ON_FULL_DEV = {FINAL_RETRAIN_ON_FULL_DEV}")
    print(f"Active aspects: {ASPECT_NAMES}")
    print(f"Aspects cache: {ASPECTS_CACHE_PATH}")

    print("\nRunning ASPECT-AWARE version with pre-extracted aspects")
    aspect_loader = PreExtractedAspectLoader(ASPECTS_CACHE_PATH)

    model = AspectAwareEncoderWithFusion(
        base_model,
        aspect_names=ASPECT_NAMES,
        fusion_type=FUSION_TYPE
    ).to(DEVICE)

    if DEVICE == "cuda":
        model.base_model.gradient_checkpointing_enable()

    if USE_SYNTHETIC_STAGE1:
        synth_ds = SyntheticTripletDataset(SYNTH_PATH)
        model = train_stage1_triplet(model, tokenizer, synth_ds, epochs=1)

    post_stage1_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    full_train_ds = AspectAwareTrackATrainDataset(DEV_TRACK_A_PATH, aspect_loader)
    train_ds, val_ds = stratified_split_aspect_dataset(full_train_ds)
    model, best_epoch = train_stage2_rank_aspect_aware(
        model, tokenizer, train_ds, val_ds, epochs=4,
        run_dir=run_dir, experiment_config=experiment_config
    )

    print(f"\n✓ Selected aspect weight after dev-split training: {model.get_aspect_weight():.4f}")
    print(f"Aspect cache stats after dev-stage access: {aspect_loader.get_stats()}")

    if FINAL_RETRAIN_ON_FULL_DEV:
        print("\nRetraining Stage 2 on the full dev set from the post-Stage-1 checkpoint...")

        # Reuse the SAME model object instead of creating a second one
        model.load_state_dict(post_stage1_state)
        model = model.to(DEVICE)
        model.train()

        # free anything we can before full-dev retraining
        del train_ds, val_ds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        epochs_for_full_retrain = best_epoch if best_epoch is not None else 2

        # safer batch size for full-dev retrain
        full_dev_bs = 2

        full_loader = DataLoader(
            full_train_ds,
            batch_size=full_dev_bs,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

        param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
        for encoder in model.aspect_encoders.values():
            param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
        param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})
        if hasattr(model, "aspect_attention"):
            param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
            param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
        if hasattr(model, "gates"):
            param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
        if hasattr(model, "concat_proj"):
            param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

        optimizer = torch.optim.AdamW(param_groups, foreach=False)
        criterion = nn.BCEWithLogitsLoss()

        for ep in range(epochs_for_full_retrain):
            loop = tqdm(full_loader, desc=f"Stage2 Full-Dev Epoch {ep+1}")
            running_loss = 0.0
            for batch_idx, batch in enumerate(loop):
                anchor_ids = batch["anchor_input_ids"].to(DEVICE)
                anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
                a_ids = batch["a_input_ids"].to(DEVICE)
                a_mask = batch["a_attention_mask"].to(DEVICE)
                b_ids = batch["b_input_ids"].to(DEVICE)
                b_mask = batch["b_attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
                a_emb = model(a_ids, a_mask, batch["a_aspects"])
                b_emb = model(b_ids, b_mask, batch["b_aspects"])

                score = F.cosine_similarity(anchor_emb, a_emb) - F.cosine_similarity(anchor_emb, b_emb)
                loss = criterion(score, labels)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                running_loss += loss.item()
                loop.set_postfix(loss=running_loss / (batch_idx + 1))

        model.eval()
        print(f"✓ Final aspect weight after full-dev retrain: {model.get_aspect_weight():.4f}")

    predict_aspect_aware_trackA(model, tokenizer, TEST_TRACK_A_PATH, OUT_TRACK_A, aspect_loader)
    build_trackB_embeddings_aspect_aware(model, tokenizer, TEST_TRACK_B_PATH, OUT_TRACK_B, aspect_loader)

    print(f"Aspect cache stats after test-time access: {aspect_loader.get_stats()}")

    sanity_checks(TEST_TRACK_A_PATH, TEST_TRACK_B_PATH, OUT_TRACK_A, OUT_TRACK_B)

    res_a = evaluate_trackA(OUT_TRACK_A, TEST_TRACK_A_LABELS)
    res_b = evaluate_trackB(OUT_TRACK_B, TEST_TRACK_B_LABELS, TEST_TRACK_B_PATH)
    print_summary(res_a, res_b)

    run_summary = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "run_name": run_name,
        "run_dir": run_dir,
        "config": experiment_config,
        "aspect_cache_stats": aspect_loader.get_stats(),
        "final_aspect_weight": model.get_aspect_weight(),
        "track_a": res_a,
        "track_b": res_b,
        "artifacts": {
            "track_a_predictions": OUT_TRACK_A,
            "track_b_embeddings": OUT_TRACK_B,
            "config_path": os.path.join(run_dir, "config.json"),
            "history_path": os.path.join(run_dir, "training_history.json"),
            "best_checkpoint_path": os.path.join(run_dir, "best_model.pt"),
        },
    }
    save_json(os.path.join(run_dir, "summary.json"), run_summary)
    append_jsonl(RESULTS_LOG_PATH, run_summary)

    print(f"Run summary saved: {os.path.join(run_dir, 'summary.json')}")
    print(f"Results log updated: {RESULTS_LOG_PATH}")



**Track A accuracy  : 69.00%Track A macro-F1  : 68.99%Track B accuracy  : 66.50%Track B macro-F1  : 66.50%**

    SEED = 42 (MAX_LEN = 256)
    RUN_NAME = 20260427_141052_aspects_action_outcome_attention_synth_seed42

*    ASPECTS_TO_USE = 'action_outcome'    FUSION_TYPE = 'attention'*

    FINAL_RETRAIN_ON_FULL_DEV = False
    USE_SYNTHETIC_STAGE1 = True
    USE_ASPECTS = True

 
**Track A accuracy  : 67.50%Track A macro-F1  : 67.50%Track B accuracy  : 65.25%Track B macro-F1  : 65.25%**

    SEED = 42 (MAX_LEN = 256)
    RUN_NAME = 20260427_181538_aspects_action_outcome_gated_synth_seed42

*    ASPECTS_TO_USE = 'action_outcome'    FUSION_TYPE = 'gated'*

    FINAL_RETRAIN_ON_FULL_DEV = False
    Active aspects: ['action', 'outcome']


**Track A accuracy  : 68.50%  Track A macro-F1  : 68.49%  Track B accuracy  : 67.25%  Track B macro-F1  : 67.25%**

    SEED = 42
    RUN_NAME = 20260427_204251_aspects_action_outcome_attention_synth_seed42
    ASPECTS_TO_USE = 'action_outcome'
    FUSION_TYPE = 'attention'
    FINAL_RETRAIN_ON_FULL_DEV = True
    Active aspects: ['action', 'outcome']

  
**Track A accuracy  : 69.25%  Track A macro-F1  : 69.22%  Track B accuracy  : 67.00%  Track B macro-F1  : 67.00%**

    SEED = 42
    RUN_NAME = 20260427_212635_aspects_action_outcome_attention_synth_seed42
    ASPECTS_TO_USE = 'action_outcome'
    FUSION_TYPE = 'attention'
    FINAL_RETRAIN_ON_FULL_DEV = False
    Active aspects: ['action', 'outcome']



**Description for the below solution**
I built an aspect-aware narrative similarity model on top of `all-roberta-large-v1`. The main idea is to encode each story in two ways: first from the full text, and second from pre-extracted narrative aspects, especially the **course of action** and the **outcome**. The aspect information is fused with attention and then added as a small residual signal to the full-text embedding, with a learnable weight.

Training is done in two stages. First, I adapt the encoder on the organizer’s synthetic triplets using triplet loss, so the model learns a narrative similarity space before seeing the small labeled dev set. Second, I fine-tune it on the SemEval dev triples as a ranking task, where the model learns to decide whether text A or text B is closer to the anchor.

This same representation is then used for both tracks: for **Track A** it compares cosine similarities between the anchor and the two candidates, and for **Track B** it outputs one embedding per story and similarity is computed in embedding space.

What is important is that I cleaned the method so it is thesis-safe: exact aspect lookup only, stratified validation split, reproducible setup, and no noisy fallback matching. Empirically, this version gives strong results on **Track A**, clearly above the provided baseline, and improves Track B compared with weaker versions, although Track B is still harder because it requires a globally coherent embedding space.

We propose an aspect-aware residual encoder for narrative similarity. Each story is represented by a pretrained full-text embedding and a fused embedding of pre-extracted narrative aspects, specifically course of action and outcome. The aspect representation is built by separately encoding each aspect text with a frozen encoder, projecting each aspect through trainable aspect-specific layers, and fusing them using attention. The fused aspect vector is added to the full-text representation through a learned residual weight. Training is performed in two stages: synthetic triplet pretraining followed by pairwise ranking fine-tuning on the SemEval development set. The resulting model is used both for Track A binary ranking decisions and for Track B embedding generation.

The Aspect-aware residual solution with the extra dataset
Below is the modified version of the code that combines the original dev data with the sample data for Stage 2 training.

# Sample data for Stage 2

In [ ]:
import os
import json
import zipfile
import random
from copy import deepcopy
from datetime import datetime, UTC
from functools import partial
from pathlib import Path
from typing import Dict, List, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SEED = 42
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "sentence-transformers/all-roberta-large-v1"
MAX_LEN = 256

STAGE1_BS = 2
STAGE2_BS = 4
PRED_BS   = 32
ENC_BS    = 16
VAL_RATIO = 0.10

USE_SYNTHETIC_STAGE1 = True
USE_ASPECTS = True
USE_EXTRA_SYNTH_STAGE2 = True

ASPECTS_TO_USE = "action_outcome"
FUSION_TYPE = "attention"
FINAL_RETRAIN_ON_FULL_DEV = True

EXTRA_SYNTH_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/sample_track_a.jsonl"
SYNTH_PATH         = "/kaggle/input/datasets/anisoaraana/similarity/synthetic_data_for_classification.jsonl"
DEV_TRACK_A_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl"
TEST_TRACK_A_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl"
TEST_TRACK_B_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl"

TEST_TRACK_A_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a_labels.jsonl"
TEST_TRACK_B_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl"

ASPECTS_CACHE_PATH = "/kaggle/input/datasets/anisoaraana/similarity/clean_extracted_aspects_including_sample.json"

OUT_DIR = "/kaggle/working/"
OUT_TRACK_A = OUT_DIR + "track_a.jsonl"
OUT_TRACK_B = OUT_DIR + "track_b.npy"

RUNS_DIR    = OUT_DIR + "runs"
RESULTS_LOG_PATH = OUT_DIR + "experiment_results.jsonl"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

    try:
        if hasattr(torch.backends.cuda, "enable_flash_sdp"):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
    except Exception:
        pass

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def make_run_name() -> str:
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    mode = "aspects" if USE_ASPECTS else "baseline"
    synth = "synth" if USE_SYNTHETIC_STAGE1 else "nosynth"
    extra = "extra" if USE_EXTRA_SYNTH_STAGE2 else "noextra"
    return f"{timestamp}_{mode}_{ASPECTS_TO_USE}_{FUSION_TYPE}_{synth}_{extra}_seed{SEED}"

def save_json(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def append_jsonl(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path: str, model: nn.Module, epoch: int, metrics: Dict[str, Any], config: Dict[str, Any]) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "metrics": metrics,
            "config": config,
        },
        path,
    )

def load_checkpoint(path: str, model: nn.Module) -> Dict[str, Any]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

def compute_binary_metrics(preds, golds):
    n = len(golds)
    if n == 0:
        return {"accuracy": 0.0, "macro_f1": 0.0, "tp": 0, "tn": 0, "fp": 0, "fn": 0}

    correct = sum(p == g for p, g in zip(preds, golds))
    accuracy = correct / n

    tp = sum(p and g for p, g in zip(preds, golds))
    tn = sum((not p) and (not g) for p, g in zip(preds, golds))
    fp = sum(p and (not g) for p, g in zip(preds, golds))
    fn = sum((not p) and g for p, g in zip(preds, golds))

    prec_a = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_a = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_a = 2 * prec_a * rec_a / (prec_a + rec_a) if (prec_a + rec_a) > 0 else 0.0

    prec_b = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_b = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_b = 2 * prec_b * rec_b / (prec_b + rec_b) if (prec_b + rec_b) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "macro_f1": (f1_a + f1_b) / 2,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

def get_aspect_info(aspects_to_use):
    if aspects_to_use == "all":
        return ["theme", "action", "outcome"], 3
    elif aspects_to_use == "theme":
        return ["theme"], 1
    elif aspects_to_use == "action":
        return ["action"], 1
    elif aspects_to_use == "outcome":
        return ["outcome"], 1
    elif aspects_to_use == "theme_action":
        return ["theme", "action"], 2
    elif aspects_to_use == "theme_outcome":
        return ["theme", "outcome"], 2
    elif aspects_to_use == "action_outcome":
        return ["action", "outcome"], 2
    else:
        return ["theme", "action", "outcome"], 3

ASPECT_NAMES, ASPECT_MULTIPLIER = get_aspect_info(ASPECTS_TO_USE)
print(f"Using aspects: {ASPECTS_TO_USE} -> {ASPECT_NAMES}")
print(f"Fusion type: {FUSION_TYPE}")

class PreExtractedAspectLoader:
    def __init__(self, cache_path: str):
        self.cache_path = cache_path
        self.aspects_cache = {}
        self.lookup_hits = 0
        self.lookup_misses = 0
        self._load_cache()
        print(f"✓ PreExtractedAspectLoader initialized with {len(self.aspects_cache)} entries")

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r", encoding="utf-8") as f:
                self.aspects_cache = json.load(f)
            print(f"  Loaded {len(self.aspects_cache)} pre-extracted aspect entries")
        else:
            print(f"  WARNING: Cache file not found at {self.cache_path}")
            print("  Will use empty aspects for all stories")

    def _normalize_text(self, text: str) -> str:
        return " ".join((text or "").split())

    def extract_aspects(self, text: str, story_id: Optional[str] = None) -> Dict[str, str]:
        norm_text = self._normalize_text(text)
        if norm_text in self.aspects_cache:
            self.lookup_hits += 1
            cached = self.aspects_cache[norm_text]
            return {
                "course_of_action": cached.get("coa", ""),
                "outcomes": cached.get("outcomes", ""),
                "abstract_theme": cached.get("theme", ""),
            }

        self.lookup_misses += 1
        return {
            "course_of_action": "",
            "outcomes": "",
            "abstract_theme": "",
        }

    def get_stats(self) -> Dict[str, Any]:
        total = self.lookup_hits + self.lookup_misses
        return {
            "hits": self.lookup_hits,
            "misses": self.lookup_misses,
            "total": total,
            "hit_rate": (self.lookup_hits / total) if total > 0 else 0.0,
        }

class AspectAwareEncoderWithFusion(nn.Module):
    def __init__(self, base_model, hidden_dim=None, aspect_dim=128,
                 aspect_names=None, fusion_type="attention", initial_aspect_weight=0.30):
        super().__init__()
        self.base_model = base_model
        self.hidden_dim = hidden_dim or base_model.config.hidden_size
        self.aspect_dim = aspect_dim
        self.aspect_names = aspect_names or ["theme", "action", "outcome"]
        self.fusion_type = fusion_type
        self.num_aspects = len(self.aspect_names)
        self.aspect_embedding_cache = {}

        if self.fusion_type == "attention":
            self.num_heads = min(4, self.num_aspects)
            self.aspect_dim = ((aspect_dim + self.num_heads - 1) // self.num_heads) * self.num_heads
            print(f"  Adjusted aspect_dim to {self.aspect_dim} for divisibility by {self.num_heads}")

        init_weight = torch.tensor(initial_aspect_weight, dtype=torch.float32).clamp(1e-4, 1 - 1e-4)
        self.aspect_weight_raw = nn.Parameter(torch.logit(init_weight))

        self.aspect_encoders = nn.ModuleDict()
        for aspect_name in self.aspect_names:
            self.aspect_encoders[aspect_name] = nn.Sequential(
                nn.Linear(self.hidden_dim, self.aspect_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.aspect_dim, self.aspect_dim)
            )

        self.aspect_text_encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.aspect_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        for param in self.aspect_text_encoder.parameters():
            param.requires_grad = False
        self.aspect_text_encoder.eval()

        if fusion_type == "attention":
            self.aspect_attention = nn.MultiheadAttention(
                embed_dim=self.aspect_dim,
                num_heads=self.num_heads,
                batch_first=True,
                dropout=0.1
            )
            self.aspect_query = nn.Parameter(torch.randn(1, 1, self.aspect_dim))
        elif fusion_type == "gated":
            self.gates = nn.Sequential(
                nn.Linear(self.aspect_dim * self.num_aspects, self.num_aspects),
                nn.Sigmoid()
            )
        elif fusion_type == "concat":
            self.concat_proj = nn.Linear(self.num_aspects * self.aspect_dim, self.aspect_dim)

        self.final_proj = nn.Sequential(
            nn.Linear(self.aspect_dim, self.hidden_dim),
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(0.1)
        )

        self.base_model.config.use_cache = False
        self.aspect_text_encoder.config.use_cache = False

        print(" AspectAwareEncoderWithFusion initialized")
        print(f"  - Aspects: {self.aspect_names}")
        print(f"  - Num aspects: {self.num_aspects}")
        print(f"  - Aspect dim: {self.aspect_dim}")
        print(f"  - Fusion type: {fusion_type}")
        print(f"  - Learnable aspect weight: {self.get_aspect_weight():.3f}")

    def encode_aspect_text(self, text: str) -> torch.Tensor:
        if not text or len(text) < 5:
            return torch.zeros(self.hidden_dim, device=next(self.base_model.parameters()).device)

        cache_key = " ".join(text.split())
        if cache_key in self.aspect_embedding_cache:
            return self.aspect_embedding_cache[cache_key].to(next(self.base_model.parameters()).device)

        inputs = self.aspect_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(next(self.base_model.parameters()).device)

        with torch.no_grad():
            outputs = self.aspect_text_encoder(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)

        self.aspect_embedding_cache[cache_key] = emb.detach().cpu()
        return emb

    def _get_aspect_text(self, aspects: Dict[str, str], aspect_name: str) -> str:
        if aspect_name == "theme":
            return aspects.get("abstract_theme", "")
        elif aspect_name == "action":
            return aspects.get("course_of_action", "")
        elif aspect_name == "outcome":
            return aspects.get("outcomes", "")
        return ""

    def _encode_single_story_aspects(self, aspects: Dict[str, str]) -> torch.Tensor:
        aspect_embeddings = []
        device = next(self.base_model.parameters()).device

        for aspect_name in self.aspect_names:
            aspect_text = self._get_aspect_text(aspects, aspect_name)
            if aspect_text:
                text_emb = self.encode_aspect_text(aspect_text)
                aspect_emb = self.aspect_encoders[aspect_name](text_emb.unsqueeze(0)).squeeze(0)
            else:
                aspect_emb = torch.zeros(self.aspect_dim, device=device)
            aspect_embeddings.append(aspect_emb)

        return torch.stack(aspect_embeddings)

    def fuse_aspects(self, aspect_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size, num_aspects, _ = aspect_embeddings.shape

        if self.fusion_type == "attention":
            query = self.aspect_query.expand(batch_size, -1, -1)
            attended, attn_weights = self.aspect_attention(query, aspect_embeddings, aspect_embeddings)
            fused = attended.squeeze(1)
            if not self.training:
                self.last_attn_weights = attn_weights.detach().cpu()
        elif self.fusion_type == "gated":
            flat = aspect_embeddings.reshape(batch_size, -1)
            gate_weights = self.gates(flat).unsqueeze(-1)
            fused = (aspect_embeddings * gate_weights).sum(dim=1)
        else:
            fused = aspect_embeddings.reshape(batch_size, -1)
            fused = self.concat_proj(fused)

        fused = self.final_proj(fused)
        fused = F.normalize(fused, p=2, dim=1)
        return fused

    def forward(self, input_ids, attention_mask, aspects: Optional[List[Dict[str, str]]] = None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        base_emb = mean_pool(outputs, attention_mask)

        if aspects is None:
            return base_emb

        if isinstance(aspects, list):
            all_aspect_embs = [self._encode_single_story_aspects(asp) for asp in aspects]
            aspect_embeddings = torch.stack(all_aspect_embs)
            aspect_contribution = self.fuse_aspects(aspect_embeddings)
        else:
            aspect_embs = self._encode_single_story_aspects(aspects)
            aspect_contribution = self.fuse_aspects(aspect_embs.unsqueeze(0))

        aspect_weight = torch.sigmoid(self.aspect_weight_raw)
        final_emb = base_emb + aspect_weight * aspect_contribution
        final_emb = F.normalize(final_emb, p=2, dim=1)
        return final_emb

    def get_aspect_weight(self):
        return torch.sigmoid(self.aspect_weight_raw).item()

    def get_attention_weights(self):
        if hasattr(self, "last_attn_weights"):
            return self.last_attn_weights
        return None

class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                anchor = obj.get("anchor_text", "")
                ta = obj.get("text_a", "")
                tb = obj.get("text_b", "")
                closer = obj.get("text_a_is_closer")
                if not anchor or not ta or not tb:
                    continue
                if len(anchor) < 20 or len(ta) < 20 or len(tb) < 20:
                    continue
                pos, neg = (ta, tb) if closer else (tb, ta)
                self.data.append((anchor.strip(), pos.strip(), neg.strip()))
        print(f"SyntheticTripletDataset: {len(self.data)} triplets")

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class AspectAwareTrackATrainDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "label": bool(obj.get("text_a_is_closer", False)),
                    "anchor_id": f"anchor_{idx}",
                    "a_id": f"a_{idx}",
                    "b_id": f"b_{idx}",
                })
        print(f"AspectAwareTrackATrainDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "label": torch.tensor(item["label"], dtype=torch.float32),
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackATestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "anchor_id": f"test_anchor_{idx}",
                    "a_id": f"test_a_{idx}",
                    "b_id": f"test_b_{idx}",
                })
        print(f"AspectAwareTrackATestDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackBTestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.texts = []
        self.ids = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.texts.append(obj["text"])
                self.ids.append(f"test_b_{idx}")
        print(f"AspectAwareTrackBTestDataset: {len(self.texts)} texts")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        return {
            "text": text,
            "aspects": self.aspect_loader.extract_aspects(text, self.ids[idx]),
            "id": self.ids[idx],
        }

def tokenize_texts(tokenizer, texts, max_len):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

def aspect_aware_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "label": torch.stack([b["label"] for b in batch]),
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_test_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_b_collate(batch, tokenizer, max_len):
    enc = tokenize_texts(tokenizer, [b["text"] for b in batch], max_len)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "aspects": [b["aspects"] for b in batch],
        "id": [b["id"] for b in batch],
    }

def stratified_split_from_labels(dataset, labels, val_ratio=VAL_RATIO, seed=SEED):
    """Return Subset train and validation datasets based on a list of labels."""
    indices = np.arange(len(labels))
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    train_idx, val_idx = next(splitter.split(indices, labels))
    train_ds = Subset(dataset, train_idx.tolist())
    val_ds = Subset(dataset, val_idx.tolist())
    print(f"Dataset split (stratified): train={len(train_ds)} val={len(val_ds)} (val_ratio={val_ratio:.2f})")
    return train_ds, val_ds

def evaluate_aspect_aware_loader(model, loader, criterion):
    model.eval()
    losses, preds, golds = [], [], []

    with torch.no_grad():
        for batch in loader:
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            losses.append(criterion(score, labels).item())

            preds.extend((score > 0).cpu().tolist())
            golds.extend(labels.bool().cpu().tolist())

    metrics = compute_binary_metrics(preds, golds)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    model.train()
    return metrics

def train_stage1_triplet(model, tokenizer, dataset, epochs=1):
    if len(dataset) == 0:
        print("No synthetic data — skipping Stage 1")
        return model

    loader = DataLoader(
        dataset,
        batch_size=STAGE1_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()

    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage1 Epoch {ep+1}")
        running = 0.0

        for anchors, positives, negatives in loop:
            texts = list(anchors) + list(positives) + list(negatives)
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(DEVICE)

            emb = model(enc["input_ids"], enc["attention_mask"], aspects=None)

            B = len(anchors)
            a, p, n = emb[:B], emb[B:2*B], emb[2*B:]
            loss = F.triplet_margin_loss(a, p, n, margin=0.5)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running += loss.item()
            loop.set_postfix(loss=running / (loop.n + 1))

    model.eval()
    print("Stage 1 finished.")
    return model

def train_stage2_rank_aspect_aware(model, tokenizer, train_dataset, val_dataset=None, epochs=2,
                                   run_dir=None, experiment_config=None):
    train_loader = DataLoader(
        train_dataset,
        batch_size=STAGE2_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=PRED_BS,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

    param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
    for encoder in model.aspect_encoders.values():
        param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
    param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
    param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})

    if hasattr(model, "aspect_attention"):
        param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
    if hasattr(model, "gates"):
        param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
    if hasattr(model, "concat_proj"):
        param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

    optimizer = torch.optim.AdamW(param_groups)
    criterion = nn.BCEWithLogitsLoss()
    model.train()

    weight_history = []
    epoch_history = []
    best_val_metrics = None
    best_checkpoint_path = None
    best_epoch = None

    if run_dir:
        ensure_dir(run_dir)
        best_checkpoint_path = os.path.join(run_dir, "best_model.pt")

    for ep in range(epochs):
        loop = tqdm(train_loader, desc=f"Stage2 Aspect Epoch {ep+1}")
        running_loss = 0.0

        for batch_idx, batch in enumerate(loop):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            loss = criterion(score, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=running_loss / (batch_idx + 1))

        current_weight = model.get_aspect_weight()
        weight_history.append(current_weight)
        print(f"  Epoch {ep+1} - Learnable aspect weight: {current_weight:.4f}")

        epoch_record = {
            "epoch": ep + 1,
            "train_loss": running_loss / max(len(train_loader), 1),
            "aspect_weight": current_weight,
        }

        if val_loader is not None:
            val_metrics = evaluate_aspect_aware_loader(model, val_loader, criterion)
            epoch_record["val"] = val_metrics
            print(
                f"  Epoch {ep+1} - Val loss: {val_metrics['loss']:.4f}  "
                f"Val acc: {val_metrics['accuracy']*100:.2f}%  "
                f"Val macro-F1: {val_metrics['macro_f1']*100:.2f}%"
            )
            if best_val_metrics is None or val_metrics["macro_f1"] > best_val_metrics["macro_f1"] or (
                val_metrics["macro_f1"] == best_val_metrics["macro_f1"] and val_metrics["loss"] < best_val_metrics["loss"]
            ):
                best_val_metrics = dict(val_metrics)
                best_epoch = ep + 1
                if best_checkpoint_path:
                    save_checkpoint(best_checkpoint_path, model, best_epoch, best_val_metrics, experiment_config or {})
                    print(f"  Saved new best checkpoint to {best_checkpoint_path}")

        epoch_history.append(epoch_record)

    if best_checkpoint_path and os.path.exists(best_checkpoint_path):
        checkpoint = load_checkpoint(best_checkpoint_path, model)
        best_epoch = checkpoint["epoch"]
        best_val_metrics = checkpoint["metrics"]
        print(
            f"Reloaded best aspect-aware checkpoint from epoch {best_epoch} "
            f"(val macro-F1={best_val_metrics['macro_f1']*100:.2f}%)"
        )

    if run_dir:
        save_json(
            os.path.join(run_dir, "training_history.json"),
            {
                "mode": "aspect_aware",
                "epochs": epoch_history,
                "best_epoch": best_epoch,
                "best_val_metrics": best_val_metrics,
                "weight_history": weight_history,
            },
        )

    model.eval()
    print(f"Stage 2 aspect-aware training finished. Final aspect weight: {model.get_aspect_weight():.4f}")
    print(f"Weight history: {[f'{w:.4f}' for w in weight_history]}")
    return model, best_epoch

def predict_aspect_aware_trackA(model, tokenizer, test_path, out_path, aspect_loader):
    ds = AspectAwareTrackATestDataset(test_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=PRED_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_test_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    preds = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict Track A with aspects"):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            preds.extend((sim_a > sim_b).cpu().tolist())

    with open(test_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line, p in zip(fin, preds):
            obj = json.loads(line)
            obj["text_a_is_closer"] = bool(p)
            fout.write(json.dumps(obj) + "\n")

    print("Track A predictions saved:", out_path)
    return preds

def build_trackB_embeddings_aspect_aware(model, tokenizer, in_path, out_npy, aspect_loader):
    ds = AspectAwareTrackBTestDataset(in_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=ENC_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_b_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    embs = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Encode Track B with aspects"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            emb = model(input_ids, attention_mask, batch["aspects"])
            embs.append(emb.cpu().numpy().astype(np.float32))

    X = np.vstack(embs)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-12, None)
    np.save(out_npy, X)
    print(f"Track B embeddings saved: {out_npy} shape={X.shape}")
    return X

def evaluate_trackA(predictions_path, gold_labels_path):
    preds, golds = [], []
    with open(predictions_path, encoding="utf-8") as fp, open(gold_labels_path, encoding="utf-8") as fg:
        for line_p, line_g in zip(fp, fg):
            preds.append(bool(json.loads(line_p)["text_a_is_closer"]))
            golds.append(bool(json.loads(line_g)["text_a_is_closer"]))

    metrics = compute_binary_metrics(preds, golds)
    metrics["n"] = len(golds)
    metrics["correct"] = sum(p == g for p, g in zip(preds, golds))

    print("\n" + "="*55)
    print("  TRACK A — TEST SET EVALUATION")
    print("="*55)
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def evaluate_trackB(embeddings_npy, gold_labels_path, test_track_b_path):
    X = np.load(embeddings_npy)
    text2idx = {}
    with open(test_track_b_path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            text2idx[json.loads(line)["text"].strip()] = idx

    correct = 0
    total = 0
    missing = 0
    golds, decisions = [], []

    with open(gold_labels_path, encoding="utf-8") as fg:
        for line in fg:
            obj = json.loads(line)
            anchor = obj["anchor_text"].strip()
            ta = obj["text_a"].strip()
            tb = obj["text_b"].strip()
            gold = bool(obj["text_a_is_closer"])

            idx_a = text2idx.get(anchor)
            idx_ta = text2idx.get(ta)
            idx_tb = text2idx.get(tb)

            if idx_a is None or idx_ta is None or idx_tb is None:
                missing += 1
                continue

            emb_a = X[idx_a]
            emb_ta = X[idx_ta]
            emb_tb = X[idx_tb]

            pred = float(np.dot(emb_a, emb_ta)) > float(np.dot(emb_a, emb_tb))
            golds.append(gold)
            decisions.append(pred)
            if pred == gold:
                correct += 1
            total += 1

    metrics = compute_binary_metrics(decisions, golds)
    metrics["n"] = total
    metrics["missing"] = missing
    metrics["correct"] = correct
    metrics["embedding_dim"] = int(X.shape[1])

    print("\n" + "="*55)
    print("  TRACK B — TEST SET EVALUATION")
    print("="*55)
    print(f"  Embedding dim      : {metrics['embedding_dim']}")
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Skipped (no embed) : {metrics['missing']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def print_summary(res_a, res_b):
    print("\n" + "="*55)
    print("  FINAL SUMMARY")
    print("="*55)
    print(f"  Track A accuracy  : {res_a['accuracy']*100:.2f}%")
    print(f"  Track A macro-F1  : {res_a['macro_f1']*100:.2f}%")
    print(f"  Track B accuracy  : {res_b['accuracy']*100:.2f}%")
    print(f"  Track B macro-F1  : {res_b['macro_f1']*100:.2f}%")
    print("-"*55)
    print("  SemEval baselines (Narrative Team CodaBench results):")
    print("    Track A : 64.25%")
    print("    Track B : 69.25%")
    print("="*55 + "\n")

def sanity_checks(test_a, test_b, out_a, out_b):
    n_ta = sum(1 for _ in open(test_a, encoding="utf-8"))
    n_oa = sum(1 for _ in open(out_a, encoding="utf-8"))
    assert n_ta == n_oa, f"Track A line mismatch: {n_ta} vs {n_oa}"

    n_tb = sum(1 for _ in open(test_b, encoding="utf-8"))
    X = np.load(out_b)
    assert X.shape[0] == n_tb, f"Track B row mismatch: {n_tb} vs {X.shape[0]}"
    assert np.isfinite(X).all(), "Track B has NaN/Inf"
    print(f"Sanity OK — TrackA lines={n_oa}  TrackB rows={X.shape[0]} dim={X.shape[1]}")

if __name__ == "__main__":
    print("="*60)
    print("NARRATIVE SIMILARITY WITH PRE-EXTRACTED ASPECTS & LEARNABLE RESIDUAL")
    print("="*60)

    set_seed(SEED)
    configure_determinism()
    ensure_dir(RUNS_DIR)

    run_name = make_run_name()
    run_dir = os.path.join(RUNS_DIR, run_name)
    ensure_dir(run_dir)

    experiment_config = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "seed": SEED,
        "device": DEVICE,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "use_synthetic_stage1": USE_SYNTHETIC_STAGE1,
        "use_aspects": USE_ASPECTS,
        "use_extra_synth_stage2": USE_EXTRA_SYNTH_STAGE2,
        "aspects_to_use": ASPECTS_TO_USE,
        "aspect_names": ASPECT_NAMES,
        "fusion_type": FUSION_TYPE,
        "final_retrain_on_full_dev": FINAL_RETRAIN_ON_FULL_DEV,
        "extra_synth_path": EXTRA_SYNTH_PATH if USE_EXTRA_SYNTH_STAGE2 else None,
    }
    save_json(os.path.join(run_dir, "config.json"), experiment_config)

    print(f"SEED = {SEED}")
    print(f"RUN_NAME = {run_name}")
    print(f"ASPECTS_TO_USE = '{ASPECTS_TO_USE}'")
    print(f"FUSION_TYPE = '{FUSION_TYPE}'")
    print(f"FINAL_RETRAIN_ON_FULL_DEV = {FINAL_RETRAIN_ON_FULL_DEV}")
    print(f"USE_EXTRA_SYNTH_STAGE2 = {USE_EXTRA_SYNTH_STAGE2}")
    print(f"Active aspects: {ASPECT_NAMES}")
    print(f"Aspects cache: {ASPECTS_CACHE_PATH}")

    print("\nRunning ASPECT-AWARE version with pre-extracted aspects")
    aspect_loader = PreExtractedAspectLoader(ASPECTS_CACHE_PATH)

    model = AspectAwareEncoderWithFusion(
        base_model,
        aspect_names=ASPECT_NAMES,
        fusion_type=FUSION_TYPE
    ).to(DEVICE)

    if DEVICE == "cuda":
        model.base_model.gradient_checkpointing_enable()

    # Stage 1: triplet training (synthetic data)
    if USE_SYNTHETIC_STAGE1:
        synth_ds = SyntheticTripletDataset(SYNTH_PATH)
        model = train_stage1_triplet(model, tokenizer, synth_ds, epochs=1)

    # Save state after Stage 1 for potential full‑dev retraining
    post_stage1_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ------------------------------------------------------------------
    # Stage 2: build combined dataset from dev + extra synthetic data
    # ------------------------------------------------------------------
    dev_dataset = AspectAwareTrackATrainDataset(DEV_TRACK_A_PATH, aspect_loader)
    print(f"Loaded dev data: {len(dev_dataset)} examples")

    extra_dataset = None
    if USE_EXTRA_SYNTH_STAGE2:
        extra_dataset = AspectAwareTrackATrainDataset(EXTRA_SYNTH_PATH, aspect_loader)
        print(f"Loaded extra synthetic data: {len(extra_dataset)} examples")

    if USE_EXTRA_SYNTH_STAGE2 and extra_dataset is not None:
        combined_dataset = ConcatDataset([dev_dataset, extra_dataset])
        # Build label list for stratified split (access underlying .data to avoid double aspect extraction)
        labels = [item["label"] for item in dev_dataset.data] + [item["label"] for item in extra_dataset.data]
        print(f"Combined dataset size: {len(combined_dataset)} examples")
    else:
        combined_dataset = dev_dataset
        labels = [item["label"] for item in dev_dataset.data]
        print(f"Using only dev data: {len(combined_dataset)} examples")

    # Stratified split on combined_dataset
    train_ds, val_ds = stratified_split_from_labels(combined_dataset, labels, val_ratio=VAL_RATIO, seed=SEED)

    model, best_epoch = train_stage2_rank_aspect_aware(
        model, tokenizer, train_ds, val_ds, epochs=4,
        run_dir=run_dir, experiment_config=experiment_config
    )

    print(f"\n✓ Selected aspect weight after dev-split training: {model.get_aspect_weight():.4f}")
    print(f"Aspect cache stats after dev-stage access: {aspect_loader.get_stats()}")

    # ------------------------------------------------------------------
    # Optional full‑dev retraining (on all dev + extra, no validation split)
    # ------------------------------------------------------------------
    if FINAL_RETRAIN_ON_FULL_DEV:
        print("\nRetraining Stage 2 on the full dev+extra set from the post-Stage-1 checkpoint...")
        model.load_state_dict(post_stage1_state)
        model = model.to(DEVICE)
        model.train()

        # free memory
        del train_ds, val_ds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Use the combined_dataset we built earlier, but without validation split
        full_train_ds = combined_dataset   # already includes dev + extra
        epochs_for_full_retrain = best_epoch if best_epoch is not None else 2
        full_dev_bs = 2

        full_loader = DataLoader(
            full_train_ds,
            batch_size=full_dev_bs,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

        param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
        for encoder in model.aspect_encoders.values():
            param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
        param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})
        if hasattr(model, "aspect_attention"):
            param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
            param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
        if hasattr(model, "gates"):
            param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
        if hasattr(model, "concat_proj"):
            param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

        optimizer = torch.optim.AdamW(param_groups, foreach=False)
        criterion = nn.BCEWithLogitsLoss()

        for ep in range(epochs_for_full_retrain):
            loop = tqdm(full_loader, desc=f"Stage2 Full-Dev Epoch {ep+1}")
            running_loss = 0.0
            for batch_idx, batch in enumerate(loop):
                anchor_ids = batch["anchor_input_ids"].to(DEVICE)
                anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
                a_ids = batch["a_input_ids"].to(DEVICE)
                a_mask = batch["a_attention_mask"].to(DEVICE)
                b_ids = batch["b_input_ids"].to(DEVICE)
                b_mask = batch["b_attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
                a_emb = model(a_ids, a_mask, batch["a_aspects"])
                b_emb = model(b_ids, b_mask, batch["b_aspects"])

                score = F.cosine_similarity(anchor_emb, a_emb) - F.cosine_similarity(anchor_emb, b_emb)
                loss = criterion(score, labels)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                running_loss += loss.item()
                loop.set_postfix(loss=running_loss / (batch_idx + 1))

        model.eval()
        print(f"✓ Final aspect weight after full-dev retrain: {model.get_aspect_weight():.4f}")

    # ------------------------------------------------------------------
    # Prediction and evaluation
    # ------------------------------------------------------------------
    predict_aspect_aware_trackA(model, tokenizer, TEST_TRACK_A_PATH, OUT_TRACK_A, aspect_loader)
    build_trackB_embeddings_aspect_aware(model, tokenizer, TEST_TRACK_B_PATH, OUT_TRACK_B, aspect_loader)

    print(f"Aspect cache stats after test-time access: {aspect_loader.get_stats()}")

    sanity_checks(TEST_TRACK_A_PATH, TEST_TRACK_B_PATH, OUT_TRACK_A, OUT_TRACK_B)

    res_a = evaluate_trackA(OUT_TRACK_A, TEST_TRACK_A_LABELS)
    res_b = evaluate_trackB(OUT_TRACK_B, TEST_TRACK_B_LABELS, TEST_TRACK_B_PATH)
    print_summary(res_a, res_b)

    run_summary = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "run_name": run_name,
        "run_dir": run_dir,
        "config": experiment_config,
        "aspect_cache_stats": aspect_loader.get_stats(),
        "final_aspect_weight": model.get_aspect_weight(),
        "track_a": res_a,
        "track_b": res_b,
        "artifacts": {
            "track_a_predictions": OUT_TRACK_A,
            "track_b_embeddings": OUT_TRACK_B,
            "config_path": os.path.join(run_dir, "config.json"),
            "history_path": os.path.join(run_dir, "training_history.json"),
            "best_checkpoint_path": os.path.join(run_dir, "best_model.pt"),
        },
    }
    save_json(os.path.join(run_dir, "summary.json"), run_summary)
    append_jsonl(RESULTS_LOG_PATH, run_summary)

    print(f"Run summary saved: {os.path.join(run_dir, 'summary.json')}")
    print(f"Results log updated: {RESULTS_LOG_PATH}")

# Add synthetic data to Stage 1

In [1]:
import os
import json
import zipfile
import random
from copy import deepcopy
from datetime import datetime, UTC
from functools import partial
from pathlib import Path
from typing import Dict, List, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SEED = 42
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

MODEL_NAME = "sentence-transformers/all-roberta-large-v1"
MAX_LEN = 256

STAGE1_BS = 2
STAGE2_BS = 4
PRED_BS   = 32
ENC_BS    = 16
VAL_RATIO = 0.10

USE_SYNTHETIC_STAGE1 = True
USE_ASPECTS = True
USE_EXTRA_SYNTH_STAGE2 = True

ASPECTS_TO_USE = "action_outcome"
FUSION_TYPE = "attention"
FINAL_RETRAIN_ON_FULL_DEV = True

EXTRA_SYNTH_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/sample_track_a.jsonl"
SYNTH_NEW_DEV_LIKE_PATH = "/kaggle/input/datasets/anisoaraana/similarity/tell_me_again_triplets_neg.jsonl"
SYNTH_PATH         = "/kaggle/input/datasets/anisoaraana/similarity/synthetic_data_for_classification.jsonl"
DEV_TRACK_A_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl"
TEST_TRACK_A_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl"
TEST_TRACK_B_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl"

TEST_TRACK_A_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a_labels.jsonl"
TEST_TRACK_B_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl"

ASPECTS_CACHE_PATH = "/kaggle/input/datasets/anisoaraana/similarity/clean_extracted_aspects_including_sample.json"

OUT_DIR = "/kaggle/working/"
OUT_TRACK_A = OUT_DIR + "track_a.jsonl"
OUT_TRACK_B = OUT_DIR + "track_b.npy"

RUNS_DIR    = OUT_DIR + "runs"
RESULTS_LOG_PATH = OUT_DIR + "experiment_results.jsonl"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

    try:
        if hasattr(torch.backends.cuda, "enable_flash_sdp"):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
    except Exception:
        pass

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def make_run_name() -> str:
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    mode = "aspects" if USE_ASPECTS else "baseline"
    synth = "synth" if USE_SYNTHETIC_STAGE1 else "nosynth"
    extra = "extra" if USE_EXTRA_SYNTH_STAGE2 else "noextra"
    return f"{timestamp}_{mode}_{ASPECTS_TO_USE}_{FUSION_TYPE}_{synth}_{extra}_seed{SEED}"

def save_json(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def append_jsonl(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path: str, model: nn.Module, epoch: int, metrics: Dict[str, Any], config: Dict[str, Any]) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "metrics": metrics,
            "config": config,
        },
        path,
    )

def load_checkpoint(path: str, model: nn.Module) -> Dict[str, Any]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

def compute_binary_metrics(preds, golds):
    n = len(golds)
    if n == 0:
        return {"accuracy": 0.0, "macro_f1": 0.0, "tp": 0, "tn": 0, "fp": 0, "fn": 0}

    correct = sum(p == g for p, g in zip(preds, golds))
    accuracy = correct / n

    tp = sum(p and g for p, g in zip(preds, golds))
    tn = sum((not p) and (not g) for p, g in zip(preds, golds))
    fp = sum(p and (not g) for p, g in zip(preds, golds))
    fn = sum((not p) and g for p, g in zip(preds, golds))

    prec_a = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_a = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_a = 2 * prec_a * rec_a / (prec_a + rec_a) if (prec_a + rec_a) > 0 else 0.0

    prec_b = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_b = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_b = 2 * prec_b * rec_b / (prec_b + rec_b) if (prec_b + rec_b) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "macro_f1": (f1_a + f1_b) / 2,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

def get_aspect_info(aspects_to_use):
    if aspects_to_use == "all":
        return ["theme", "action", "outcome"], 3
    elif aspects_to_use == "theme":
        return ["theme"], 1
    elif aspects_to_use == "action":
        return ["action"], 1
    elif aspects_to_use == "outcome":
        return ["outcome"], 1
    elif aspects_to_use == "theme_action":
        return ["theme", "action"], 2
    elif aspects_to_use == "theme_outcome":
        return ["theme", "outcome"], 2
    elif aspects_to_use == "action_outcome":
        return ["action", "outcome"], 2
    else:
        return ["theme", "action", "outcome"], 3

ASPECT_NAMES, ASPECT_MULTIPLIER = get_aspect_info(ASPECTS_TO_USE)
print(f"Using aspects: {ASPECTS_TO_USE} -> {ASPECT_NAMES}")
print(f"Fusion type: {FUSION_TYPE}")

class PreExtractedAspectLoader:
    def __init__(self, cache_path: str):
        self.cache_path = cache_path
        self.aspects_cache = {}
        self.lookup_hits = 0
        self.lookup_misses = 0
        self._load_cache()
        print(f"✓ PreExtractedAspectLoader initialized with {len(self.aspects_cache)} entries")

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r", encoding="utf-8") as f:
                self.aspects_cache = json.load(f)
            print(f"  Loaded {len(self.aspects_cache)} pre-extracted aspect entries")
        else:
            print(f"  WARNING: Cache file not found at {self.cache_path}")
            print("  Will use empty aspects for all stories")

    def _normalize_text(self, text: str) -> str:
        return " ".join((text or "").split())

    def extract_aspects(self, text: str, story_id: Optional[str] = None) -> Dict[str, str]:
        norm_text = self._normalize_text(text)
        if norm_text in self.aspects_cache:
            self.lookup_hits += 1
            cached = self.aspects_cache[norm_text]
            return {
                "course_of_action": cached.get("coa", ""),
                "outcomes": cached.get("outcomes", ""),
                "abstract_theme": cached.get("theme", ""),
            }

        self.lookup_misses += 1
        return {
            "course_of_action": "",
            "outcomes": "",
            "abstract_theme": "",
        }

    def get_stats(self) -> Dict[str, Any]:
        total = self.lookup_hits + self.lookup_misses
        return {
            "hits": self.lookup_hits,
            "misses": self.lookup_misses,
            "total": total,
            "hit_rate": (self.lookup_hits / total) if total > 0 else 0.0,
        }

class AspectAwareEncoderWithFusion(nn.Module):
    def __init__(self, base_model, hidden_dim=None, aspect_dim=128,
                 aspect_names=None, fusion_type="attention", initial_aspect_weight=0.30):
        super().__init__()
        self.base_model = base_model
        self.hidden_dim = hidden_dim or base_model.config.hidden_size
        self.aspect_dim = aspect_dim
        self.aspect_names = aspect_names or ["theme", "action", "outcome"]
        self.fusion_type = fusion_type
        self.num_aspects = len(self.aspect_names)
        self.aspect_embedding_cache = {}

        if self.fusion_type == "attention":
            self.num_heads = min(4, self.num_aspects)
            self.aspect_dim = ((aspect_dim + self.num_heads - 1) // self.num_heads) * self.num_heads
            print(f"  Adjusted aspect_dim to {self.aspect_dim} for divisibility by {self.num_heads}")

        init_weight = torch.tensor(initial_aspect_weight, dtype=torch.float32).clamp(1e-4, 1 - 1e-4)
        self.aspect_weight_raw = nn.Parameter(torch.logit(init_weight))

        self.aspect_encoders = nn.ModuleDict()
        for aspect_name in self.aspect_names:
            self.aspect_encoders[aspect_name] = nn.Sequential(
                nn.Linear(self.hidden_dim, self.aspect_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.aspect_dim, self.aspect_dim)
            )

        self.aspect_text_encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.aspect_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        for param in self.aspect_text_encoder.parameters():
            param.requires_grad = False
        self.aspect_text_encoder.eval()

        if fusion_type == "attention":
            self.aspect_attention = nn.MultiheadAttention(
                embed_dim=self.aspect_dim,
                num_heads=self.num_heads,
                batch_first=True,
                dropout=0.1
            )
            self.aspect_query = nn.Parameter(torch.randn(1, 1, self.aspect_dim))
        elif fusion_type == "gated":
            self.gates = nn.Sequential(
                nn.Linear(self.aspect_dim * self.num_aspects, self.num_aspects),
                nn.Sigmoid()
            )
        elif fusion_type == "concat":
            self.concat_proj = nn.Linear(self.num_aspects * self.aspect_dim, self.aspect_dim)

        self.final_proj = nn.Sequential(
            nn.Linear(self.aspect_dim, self.hidden_dim),
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(0.1)
        )

        self.base_model.config.use_cache = False
        self.aspect_text_encoder.config.use_cache = False

        print(" AspectAwareEncoderWithFusion initialized")
        print(f"  - Aspects: {self.aspect_names}")
        print(f"  - Num aspects: {self.num_aspects}")
        print(f"  - Aspect dim: {self.aspect_dim}")
        print(f"  - Fusion type: {fusion_type}")
        print(f"  - Learnable aspect weight: {self.get_aspect_weight():.3f}")

    def encode_aspect_text(self, text: str) -> torch.Tensor:
        if not text or len(text) < 5:
            return torch.zeros(self.hidden_dim, device=next(self.base_model.parameters()).device)

        cache_key = " ".join(text.split())
        if cache_key in self.aspect_embedding_cache:
            return self.aspect_embedding_cache[cache_key].to(next(self.base_model.parameters()).device)

        inputs = self.aspect_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(next(self.base_model.parameters()).device)

        with torch.no_grad():
            outputs = self.aspect_text_encoder(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)

        self.aspect_embedding_cache[cache_key] = emb.detach().cpu()
        return emb

    def _get_aspect_text(self, aspects: Dict[str, str], aspect_name: str) -> str:
        if aspect_name == "theme":
            return aspects.get("abstract_theme", "")
        elif aspect_name == "action":
            return aspects.get("course_of_action", "")
        elif aspect_name == "outcome":
            return aspects.get("outcomes", "")
        return ""

    def _encode_single_story_aspects(self, aspects: Dict[str, str]) -> torch.Tensor:
        aspect_embeddings = []
        device = next(self.base_model.parameters()).device

        for aspect_name in self.aspect_names:
            aspect_text = self._get_aspect_text(aspects, aspect_name)
            if aspect_text:
                text_emb = self.encode_aspect_text(aspect_text)
                aspect_emb = self.aspect_encoders[aspect_name](text_emb.unsqueeze(0)).squeeze(0)
            else:
                aspect_emb = torch.zeros(self.aspect_dim, device=device)
            aspect_embeddings.append(aspect_emb)

        return torch.stack(aspect_embeddings)

    def fuse_aspects(self, aspect_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size, num_aspects, _ = aspect_embeddings.shape

        if self.fusion_type == "attention":
            query = self.aspect_query.expand(batch_size, -1, -1)
            attended, attn_weights = self.aspect_attention(query, aspect_embeddings, aspect_embeddings)
            fused = attended.squeeze(1)
            if not self.training:
                self.last_attn_weights = attn_weights.detach().cpu()
        elif self.fusion_type == "gated":
            flat = aspect_embeddings.reshape(batch_size, -1)
            gate_weights = self.gates(flat).unsqueeze(-1)
            fused = (aspect_embeddings * gate_weights).sum(dim=1)
        else:
            fused = aspect_embeddings.reshape(batch_size, -1)
            fused = self.concat_proj(fused)

        fused = self.final_proj(fused)
        fused = F.normalize(fused, p=2, dim=1)
        return fused

    def forward(self, input_ids, attention_mask, aspects: Optional[List[Dict[str, str]]] = None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        base_emb = mean_pool(outputs, attention_mask)

        if aspects is None:
            return base_emb

        if isinstance(aspects, list):
            all_aspect_embs = [self._encode_single_story_aspects(asp) for asp in aspects]
            aspect_embeddings = torch.stack(all_aspect_embs)
            aspect_contribution = self.fuse_aspects(aspect_embeddings)
        else:
            aspect_embs = self._encode_single_story_aspects(aspects)
            aspect_contribution = self.fuse_aspects(aspect_embs.unsqueeze(0))

        aspect_weight = torch.sigmoid(self.aspect_weight_raw)
        final_emb = base_emb + aspect_weight * aspect_contribution
        final_emb = F.normalize(final_emb, p=2, dim=1)
        return final_emb

    def get_aspect_weight(self):
        return torch.sigmoid(self.aspect_weight_raw).item()

    def get_attention_weights(self):
        if hasattr(self, "last_attn_weights"):
            return self.last_attn_weights
        return None

class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                anchor = obj.get("anchor_text", "")
                ta = obj.get("text_a", "")
                tb = obj.get("text_b", "")
                closer = obj.get("text_a_is_closer")
                if not anchor or not ta or not tb:
                    continue
                if len(anchor) < 20 or len(ta) < 20 or len(tb) < 20:
                    continue
                pos, neg = (ta, tb) if closer else (tb, ta)
                self.data.append((anchor.strip(), pos.strip(), neg.strip()))
        print(f"SyntheticTripletDataset: {len(self.data)} triplets")

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

class AspectAwareTrackATrainDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "label": bool(obj.get("text_a_is_closer", False)),
                    "anchor_id": f"anchor_{idx}",
                    "a_id": f"a_{idx}",
                    "b_id": f"b_{idx}",
                })
        print(f"AspectAwareTrackATrainDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "label": torch.tensor(item["label"], dtype=torch.float32),
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackATestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.data.append({
                    "anchor": obj["anchor_text"],
                    "text_a": obj["text_a"],
                    "text_b": obj["text_b"],
                    "anchor_id": f"test_anchor_{idx}",
                    "a_id": f"test_a_{idx}",
                    "b_id": f"test_b_{idx}",
                })
        print(f"AspectAwareTrackATestDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

class AspectAwareTrackBTestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.texts = []
        self.ids = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.texts.append(obj["text"])
                self.ids.append(f"test_b_{idx}")
        print(f"AspectAwareTrackBTestDataset: {len(self.texts)} texts")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        return {
            "text": text,
            "aspects": self.aspect_loader.extract_aspects(text, self.ids[idx]),
            "id": self.ids[idx],
        }

def tokenize_texts(tokenizer, texts, max_len):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

def aspect_aware_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "label": torch.stack([b["label"] for b in batch]),
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_test_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_b_collate(batch, tokenizer, max_len):
    enc = tokenize_texts(tokenizer, [b["text"] for b in batch], max_len)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "aspects": [b["aspects"] for b in batch],
        "id": [b["id"] for b in batch],
    }

def stratified_split_from_labels(dataset, labels, val_ratio=VAL_RATIO, seed=SEED):
    """Return Subset train and validation datasets based on a list of labels."""
    indices = np.arange(len(labels))
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    train_idx, val_idx = next(splitter.split(indices, labels))
    train_ds = Subset(dataset, train_idx.tolist())
    val_ds = Subset(dataset, val_idx.tolist())
    print(f"Dataset split (stratified): train={len(train_ds)} val={len(val_ds)} (val_ratio={val_ratio:.2f})")
    return train_ds, val_ds

def evaluate_aspect_aware_loader(model, loader, criterion):
    model.eval()
    losses, preds, golds = [], [], []

    with torch.no_grad():
        for batch in loader:
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            losses.append(criterion(score, labels).item())

            preds.extend((score > 0).cpu().tolist())
            golds.extend(labels.bool().cpu().tolist())

    metrics = compute_binary_metrics(preds, golds)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    model.train()
    return metrics

def train_stage1_triplet(model, tokenizer, dataset, epochs=1):
    if len(dataset) == 0:
        print("No synthetic data — skipping Stage 1")
        return model

    loader = DataLoader(
        dataset,
        batch_size=STAGE1_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()

    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage1 Epoch {ep+1}")
        running = 0.0

        for anchors, positives, negatives in loop:
            texts = list(anchors) + list(positives) + list(negatives)
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(DEVICE)

            emb = model(enc["input_ids"], enc["attention_mask"], aspects=None)

            B = len(anchors)
            a, p, n = emb[:B], emb[B:2*B], emb[2*B:]
            loss = F.triplet_margin_loss(a, p, n, margin=0.5)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running += loss.item()
            loop.set_postfix(loss=running / (loop.n + 1))

    model.eval()
    print("Stage 1 finished.")
    return model

def train_stage2_rank_aspect_aware(model, tokenizer, train_dataset, val_dataset=None, epochs=2,
                                   run_dir=None, experiment_config=None):
    train_loader = DataLoader(
        train_dataset,
        batch_size=STAGE2_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=PRED_BS,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

    param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
    for encoder in model.aspect_encoders.values():
        param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
    param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
    param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})

    if hasattr(model, "aspect_attention"):
        param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
    if hasattr(model, "gates"):
        param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
    if hasattr(model, "concat_proj"):
        param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

    optimizer = torch.optim.AdamW(param_groups)
    criterion = nn.BCEWithLogitsLoss()
    model.train()

    weight_history = []
    epoch_history = []
    best_val_metrics = None
    best_checkpoint_path = None
    best_epoch = None

    if run_dir:
        ensure_dir(run_dir)
        best_checkpoint_path = os.path.join(run_dir, "best_model.pt")

    for ep in range(epochs):
        loop = tqdm(train_loader, desc=f"Stage2 Aspect Epoch {ep+1}")
        running_loss = 0.0

        for batch_idx, batch in enumerate(loop):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            loss = criterion(score, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=running_loss / (batch_idx + 1))

        current_weight = model.get_aspect_weight()
        weight_history.append(current_weight)
        print(f"  Epoch {ep+1} - Learnable aspect weight: {current_weight:.4f}")

        epoch_record = {
            "epoch": ep + 1,
            "train_loss": running_loss / max(len(train_loader), 1),
            "aspect_weight": current_weight,
        }

        if val_loader is not None:
            val_metrics = evaluate_aspect_aware_loader(model, val_loader, criterion)
            epoch_record["val"] = val_metrics
            print(
                f"  Epoch {ep+1} - Val loss: {val_metrics['loss']:.4f}  "
                f"Val acc: {val_metrics['accuracy']*100:.2f}%  "
                f"Val macro-F1: {val_metrics['macro_f1']*100:.2f}%"
            )
            if best_val_metrics is None or val_metrics["macro_f1"] > best_val_metrics["macro_f1"] or (
                val_metrics["macro_f1"] == best_val_metrics["macro_f1"] and val_metrics["loss"] < best_val_metrics["loss"]
            ):
                best_val_metrics = dict(val_metrics)
                best_epoch = ep + 1
                if best_checkpoint_path:
                    save_checkpoint(best_checkpoint_path, model, best_epoch, best_val_metrics, experiment_config or {})
                    print(f"  Saved new best checkpoint to {best_checkpoint_path}")

        epoch_history.append(epoch_record)

    if best_checkpoint_path and os.path.exists(best_checkpoint_path):
        checkpoint = load_checkpoint(best_checkpoint_path, model)
        best_epoch = checkpoint["epoch"]
        best_val_metrics = checkpoint["metrics"]
        print(
            f"Reloaded best aspect-aware checkpoint from epoch {best_epoch} "
            f"(val macro-F1={best_val_metrics['macro_f1']*100:.2f}%)"
        )

    if run_dir:
        save_json(
            os.path.join(run_dir, "training_history.json"),
            {
                "mode": "aspect_aware",
                "epochs": epoch_history,
                "best_epoch": best_epoch,
                "best_val_metrics": best_val_metrics,
                "weight_history": weight_history,
            },
        )

    model.eval()
    print(f"Stage 2 aspect-aware training finished. Final aspect weight: {model.get_aspect_weight():.4f}")
    print(f"Weight history: {[f'{w:.4f}' for w in weight_history]}")
    return model, best_epoch

def predict_aspect_aware_trackA(model, tokenizer, test_path, out_path, aspect_loader):
    ds = AspectAwareTrackATestDataset(test_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=PRED_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_test_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    preds = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict Track A with aspects"):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            preds.extend((sim_a > sim_b).cpu().tolist())

    with open(test_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line, p in zip(fin, preds):
            obj = json.loads(line)
            obj["text_a_is_closer"] = bool(p)
            fout.write(json.dumps(obj) + "\n")

    print("Track A predictions saved:", out_path)
    return preds

def build_trackB_embeddings_aspect_aware(model, tokenizer, in_path, out_npy, aspect_loader):
    ds = AspectAwareTrackBTestDataset(in_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=ENC_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_b_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    embs = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Encode Track B with aspects"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            emb = model(input_ids, attention_mask, batch["aspects"])
            embs.append(emb.cpu().numpy().astype(np.float32))

    X = np.vstack(embs)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-12, None)
    np.save(out_npy, X)
    print(f"Track B embeddings saved: {out_npy} shape={X.shape}")
    return X

def evaluate_trackA(predictions_path, gold_labels_path):
    preds, golds = [], []
    with open(predictions_path, encoding="utf-8") as fp, open(gold_labels_path, encoding="utf-8") as fg:
        for line_p, line_g in zip(fp, fg):
            preds.append(bool(json.loads(line_p)["text_a_is_closer"]))
            golds.append(bool(json.loads(line_g)["text_a_is_closer"]))

    metrics = compute_binary_metrics(preds, golds)
    metrics["n"] = len(golds)
    metrics["correct"] = sum(p == g for p, g in zip(preds, golds))

    print("\n" + "="*55)
    print("  TRACK A — TEST SET EVALUATION")
    print("="*55)
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def evaluate_trackB(embeddings_npy, gold_labels_path, test_track_b_path):
    X = np.load(embeddings_npy)
    text2idx = {}
    with open(test_track_b_path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            text2idx[json.loads(line)["text"].strip()] = idx

    correct = 0
    total = 0
    missing = 0
    golds, decisions = [], []

    with open(gold_labels_path, encoding="utf-8") as fg:
        for line in fg:
            obj = json.loads(line)
            anchor = obj["anchor_text"].strip()
            ta = obj["text_a"].strip()
            tb = obj["text_b"].strip()
            gold = bool(obj["text_a_is_closer"])

            idx_a = text2idx.get(anchor)
            idx_ta = text2idx.get(ta)
            idx_tb = text2idx.get(tb)

            if idx_a is None or idx_ta is None or idx_tb is None:
                missing += 1
                continue

            emb_a = X[idx_a]
            emb_ta = X[idx_ta]
            emb_tb = X[idx_tb]

            pred = float(np.dot(emb_a, emb_ta)) > float(np.dot(emb_a, emb_tb))
            golds.append(gold)
            decisions.append(pred)
            if pred == gold:
                correct += 1
            total += 1

    metrics = compute_binary_metrics(decisions, golds)
    metrics["n"] = total
    metrics["missing"] = missing
    metrics["correct"] = correct
    metrics["embedding_dim"] = int(X.shape[1])

    print("\n" + "="*55)
    print("  TRACK B — TEST SET EVALUATION")
    print("="*55)
    print(f"  Embedding dim      : {metrics['embedding_dim']}")
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Skipped (no embed) : {metrics['missing']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def print_summary(res_a, res_b):
    print("\n" + "="*55)
    print("  FINAL SUMMARY")
    print("="*55)
    print(f"  Track A accuracy  : {res_a['accuracy']*100:.2f}%")
    print(f"  Track A macro-F1  : {res_a['macro_f1']*100:.2f}%")
    print(f"  Track B accuracy  : {res_b['accuracy']*100:.2f}%")
    print(f"  Track B macro-F1  : {res_b['macro_f1']*100:.2f}%")
    print("-"*55)
    print("  SemEval baselines (Narrative Team CodaBench results):")
    print("    Track A : 64.25%")
    print("    Track B : 69.25%")
    print("="*55 + "\n")

def sanity_checks(test_a, test_b, out_a, out_b):
    n_ta = sum(1 for _ in open(test_a, encoding="utf-8"))
    n_oa = sum(1 for _ in open(out_a, encoding="utf-8"))
    assert n_ta == n_oa, f"Track A line mismatch: {n_ta} vs {n_oa}"

    n_tb = sum(1 for _ in open(test_b, encoding="utf-8"))
    X = np.load(out_b)
    assert X.shape[0] == n_tb, f"Track B row mismatch: {n_tb} vs {X.shape[0]}"
    assert np.isfinite(X).all(), "Track B has NaN/Inf"
    print(f"Sanity OK — TrackA lines={n_oa}  TrackB rows={X.shape[0]} dim={X.shape[1]}")

if __name__ == "__main__":
    print("="*60)
    print("NARRATIVE SIMILARITY WITH PRE-EXTRACTED ASPECTS & LEARNABLE RESIDUAL")
    print("="*60)

    set_seed(SEED)
    configure_determinism()
    ensure_dir(RUNS_DIR)

    run_name = make_run_name()
    run_dir = os.path.join(RUNS_DIR, run_name)
    ensure_dir(run_dir)

    experiment_config = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "seed": SEED,
        "device": DEVICE,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "use_synthetic_stage1": USE_SYNTHETIC_STAGE1,
        "use_aspects": USE_ASPECTS,
        "use_extra_synth_stage2": USE_EXTRA_SYNTH_STAGE2,
        "aspects_to_use": ASPECTS_TO_USE,
        "aspect_names": ASPECT_NAMES,
        "fusion_type": FUSION_TYPE,
        "final_retrain_on_full_dev": FINAL_RETRAIN_ON_FULL_DEV,
        "extra_synth_path": EXTRA_SYNTH_PATH if USE_EXTRA_SYNTH_STAGE2 else None,
    }
    save_json(os.path.join(run_dir, "config.json"), experiment_config)

    print(f"SEED = {SEED}")
    print(f"RUN_NAME = {run_name}")
    print(f"ASPECTS_TO_USE = '{ASPECTS_TO_USE}'")
    print(f"FUSION_TYPE = '{FUSION_TYPE}'")
    print(f"FINAL_RETRAIN_ON_FULL_DEV = {FINAL_RETRAIN_ON_FULL_DEV}")
    print(f"USE_EXTRA_SYNTH_STAGE2 = {USE_EXTRA_SYNTH_STAGE2}")
    print(f"Active aspects: {ASPECT_NAMES}")
    print(f"Aspects cache: {ASPECTS_CACHE_PATH}")

    print("\nRunning ASPECT-AWARE version with pre-extracted aspects")
    aspect_loader = PreExtractedAspectLoader(ASPECTS_CACHE_PATH)

    model = AspectAwareEncoderWithFusion(
        base_model,
        aspect_names=ASPECT_NAMES,
        fusion_type=FUSION_TYPE
    ).to(DEVICE)

    if DEVICE == "cuda":
        model.base_model.gradient_checkpointing_enable()

    # Stage 1: triplet training (synthetic data)
    # if USE_SYNTHETIC_STAGE1:
    #     synth_ds = SyntheticTripletDataset(SYNTH_PATH)
    #     model = train_stage1_triplet(model, tokenizer, synth_ds, epochs=1)
    if USE_SYNTHETIC_STAGE1:
        synth_paths = [SYNTH_PATH, SYNTH_NEW_DEV_LIKE_PATH]
        synth_datasets = []
        for p in synth_paths:
            if os.path.exists(p):
                print(f"Loading synthetic triplets from {p}")
                synth_datasets.append(SyntheticTripletDataset(p))
            else:
                print(f"Warning: synthetic file not found, skipping {p}")
    
        if synth_datasets:
            # Combine all loaded synthetic datasets
            combined_synth_ds = ConcatDataset(synth_datasets) if len(synth_datasets) > 1 else synth_datasets[0]
            print(f"Total synthetic triplets after combining: {len(combined_synth_ds)}")
            model = train_stage1_triplet(model, tokenizer, combined_synth_ds, epochs=1)
        else:
            print("No synthetic data — skipping Stage 1")
    else:
        print("Synthetic stage 1 disabled (USE_SYNTHETIC_STAGE1=False)")

    # Save state after Stage 1 for potential full‑dev retraining
    post_stage1_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ------------------------------------------------------------------
    # Stage 2: build combined dataset from dev + extra synthetic data
    # ------------------------------------------------------------------
    dev_dataset = AspectAwareTrackATrainDataset(DEV_TRACK_A_PATH, aspect_loader)
    print(f"Loaded dev data: {len(dev_dataset)} examples")

    extra_dataset = None
    if USE_EXTRA_SYNTH_STAGE2:
        extra_dataset = AspectAwareTrackATrainDataset(EXTRA_SYNTH_PATH, aspect_loader)
        print(f"Loaded extra synthetic data: {len(extra_dataset)} examples")

    if USE_EXTRA_SYNTH_STAGE2 and extra_dataset is not None:
        combined_dataset = ConcatDataset([dev_dataset, extra_dataset])
        # Build label list for stratified split (access underlying .data to avoid double aspect extraction)
        labels = [item["label"] for item in dev_dataset.data] + [item["label"] for item in extra_dataset.data]
        print(f"Combined dataset size: {len(combined_dataset)} examples")
    else:
        combined_dataset = dev_dataset
        labels = [item["label"] for item in dev_dataset.data]
        print(f"Using only dev data: {len(combined_dataset)} examples")

    # Stratified split on combined_dataset
    train_ds, val_ds = stratified_split_from_labels(combined_dataset, labels, val_ratio=VAL_RATIO, seed=SEED)

    model, best_epoch = train_stage2_rank_aspect_aware(
        model, tokenizer, train_ds, val_ds, epochs=4,
        run_dir=run_dir, experiment_config=experiment_config
    )

    print(f"\n✓ Selected aspect weight after dev-split training: {model.get_aspect_weight():.4f}")
    print(f"Aspect cache stats after dev-stage access: {aspect_loader.get_stats()}")

    # ------------------------------------------------------------------
    # Optional full‑dev retraining (on all dev + extra, no validation split)
    # ------------------------------------------------------------------
    if FINAL_RETRAIN_ON_FULL_DEV:
        print("\nRetraining Stage 2 on the full dev+extra set from the post-Stage-1 checkpoint...")
        model.load_state_dict(post_stage1_state)
        model = model.to(DEVICE)
        model.train()

        # free memory
        del train_ds, val_ds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Use the combined_dataset we built earlier, but without validation split
        full_train_ds = combined_dataset   # already includes dev + extra
        epochs_for_full_retrain = best_epoch if best_epoch is not None else 2
        full_dev_bs = 2

        full_loader = DataLoader(
            full_train_ds,
            batch_size=full_dev_bs,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

        param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
        for encoder in model.aspect_encoders.values():
            param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
        param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})
        if hasattr(model, "aspect_attention"):
            param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
            param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
        if hasattr(model, "gates"):
            param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
        if hasattr(model, "concat_proj"):
            param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

        optimizer = torch.optim.AdamW(param_groups, foreach=False)
        criterion = nn.BCEWithLogitsLoss()

        for ep in range(epochs_for_full_retrain):
            loop = tqdm(full_loader, desc=f"Stage2 Full-Dev Epoch {ep+1}")
            running_loss = 0.0
            for batch_idx, batch in enumerate(loop):
                anchor_ids = batch["anchor_input_ids"].to(DEVICE)
                anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
                a_ids = batch["a_input_ids"].to(DEVICE)
                a_mask = batch["a_attention_mask"].to(DEVICE)
                b_ids = batch["b_input_ids"].to(DEVICE)
                b_mask = batch["b_attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
                a_emb = model(a_ids, a_mask, batch["a_aspects"])
                b_emb = model(b_ids, b_mask, batch["b_aspects"])

                score = F.cosine_similarity(anchor_emb, a_emb) - F.cosine_similarity(anchor_emb, b_emb)
                loss = criterion(score, labels)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                running_loss += loss.item()
                loop.set_postfix(loss=running_loss / (batch_idx + 1))

        model.eval()
        print(f"✓ Final aspect weight after full-dev retrain: {model.get_aspect_weight():.4f}")

    # ------------------------------------------------------------------
    # Prediction and evaluation
    # ------------------------------------------------------------------
    predict_aspect_aware_trackA(model, tokenizer, TEST_TRACK_A_PATH, OUT_TRACK_A, aspect_loader)
    build_trackB_embeddings_aspect_aware(model, tokenizer, TEST_TRACK_B_PATH, OUT_TRACK_B, aspect_loader)

    print(f"Aspect cache stats after test-time access: {aspect_loader.get_stats()}")

    sanity_checks(TEST_TRACK_A_PATH, TEST_TRACK_B_PATH, OUT_TRACK_A, OUT_TRACK_B)

    res_a = evaluate_trackA(OUT_TRACK_A, TEST_TRACK_A_LABELS)
    res_b = evaluate_trackB(OUT_TRACK_B, TEST_TRACK_B_LABELS, TEST_TRACK_B_PATH)
    print_summary(res_a, res_b)

    run_summary = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "run_name": run_name,
        "run_dir": run_dir,
        "config": experiment_config,
        "aspect_cache_stats": aspect_loader.get_stats(),
        "final_aspect_weight": model.get_aspect_weight(),
        "track_a": res_a,
        "track_b": res_b,
        "artifacts": {
            "track_a_predictions": OUT_TRACK_A,
            "track_b_embeddings": OUT_TRACK_B,
            "config_path": os.path.join(run_dir, "config.json"),
            "history_path": os.path.join(run_dir, "training_history.json"),
            "best_checkpoint_path": os.path.join(run_dir, "best_model.pt"),
        },
    }
    save_json(os.path.join(run_dir, "summary.json"), run_summary)
    append_jsonl(RESULTS_LOG_PATH, run_summary)

    print(f"Run summary saved: {os.path.join(run_dir, 'summary.json')}")
    print(f"Results log updated: {RESULTS_LOG_PATH}")

Device: cuda


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

✓ Final aspect weight after full-dev retrain: 0.2919
AspectAwareTrackATestDataset: 400 rows


Predict Track A with aspects: 100%|██████████| 13/13 [01:53<00:00,  8.73s/it]


Track A predictions saved: /kaggle/working/track_a.jsonl
AspectAwareTrackBTestDataset: 849 texts


Encode Track B with aspects: 100%|██████████| 54/54 [01:24<00:00,  1.56s/it]

Track B embeddings saved: /kaggle/working/track_b.npy shape=(849, 1024)
Aspect cache stats after test-time access: {'hits': 5634, 'misses': 0, 'total': 5634, 'hit_rate': 1.0}
Sanity OK — TrackA lines=400  TrackB rows=849 dim=1024

  TRACK A — TEST SET EVALUATION
  Total triples      : 400
  Correct            : 244
  Accuracy           : 61.00%
  Macro F1           : 60.98%


  TRACK B — TEST SET EVALUATION
  Embedding dim      : 1024
  Total triples      : 400
  Skipped (no embed) : 0
  Correct            : 247
  Accuracy           : 61.75%
  Macro F1           : 61.74%


  FINAL SUMMARY
  Track A accuracy  : 61.00%
  Track A macro-F1  : 60.98%
  Track B accuracy  : 61.75%
  Track B macro-F1  : 61.74%
-------------------------------------------------------
  SemEval baselines (Narrative Team CodaBench results):
    Track A : 64.25%
    Track B : 69.25%

Run summary saved: /kaggle/working/runs/20260504_085905_aspects_action_outcome_attention_synth_extra_seed42/summary.json
Results log 

# Try models: "BAAI/bge-large-en-v1.5", "intfloat/e5-large-v2"

In [ ]:
import os
import json
import zipfile
import random
from copy import deepcopy
from datetime import datetime, UTC
from functools import partial
from pathlib import Path
from typing import Dict, List, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

SEED = 42
STRICT_DETERMINISM = os.environ.get("STRICT_DETERMINISM", "1").strip().lower() not in {"0", "false", "no"}
DL_NUM_WORKERS = int(os.environ.get("DL_NUM_WORKERS", 0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ========== CHANGED: model and max length ==========
MODEL_NAME = "intfloat/e5-large-v2"
MAX_LEN = 512
# ===================================================

STAGE1_BS = 2
STAGE2_BS = 4
PRED_BS   = 32
ENC_BS    = 16
VAL_RATIO = 0.10

USE_SYNTHETIC_STAGE1 = True
USE_ASPECTS = True
USE_EXTRA_SYNTH_STAGE2 = True

ASPECTS_TO_USE = "action_outcome"
FUSION_TYPE = "attention"
FINAL_RETRAIN_ON_FULL_DEV = True

EXTRA_SYNTH_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/sample_track_a.jsonl"
SYNTH_PATH         = "/kaggle/input/datasets/anisoaraana/similarity/synthetic_data_for_classification.jsonl"
DEV_TRACK_A_PATH   = "/kaggle/input/datasets/anisoaraana/similarity/dev_track_a.jsonl"
TEST_TRACK_A_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a.jsonl"
TEST_TRACK_B_PATH  = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b.jsonl"

TEST_TRACK_A_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a_labels.jsonl"
TEST_TRACK_B_LABELS = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl"

ASPECTS_CACHE_PATH = "/kaggle/input/datasets/anisoaraana/similarity/clean_extracted_aspects_including_sample.json"

OUT_DIR = "/kaggle/working/"
OUT_TRACK_A = OUT_DIR + "track_a.jsonl"
OUT_TRACK_B = OUT_DIR + "track_b.npy"

RUNS_DIR    = OUT_DIR + "runs"
RESULTS_LOG_PATH = OUT_DIR + "experiment_results.jsonl"

# ========== ADDED: text preprocessing ==========
def prep_text(x: str) -> str:
    if x is None:
        return ""
    return "query" + str(x).strip()
    
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

    try:
        if hasattr(torch.backends.cuda, "enable_flash_sdp"):
            torch.backends.cuda.enable_flash_sdp(False)
        if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
            torch.backends.cuda.enable_mem_efficient_sdp(False)
    except Exception:
        pass

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def make_run_name() -> str:
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    mode = "aspects" if USE_ASPECTS else "baseline"
    synth = "synth" if USE_SYNTHETIC_STAGE1 else "nosynth"
    extra = "extra" if USE_EXTRA_SYNTH_STAGE2 else "noextra"
    return f"{timestamp}_{mode}_{ASPECTS_TO_USE}_{FUSION_TYPE}_{synth}_{extra}_seed{SEED}"

def save_json(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def append_jsonl(path: str, payload: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path: str, model: nn.Module, epoch: int, metrics: Dict[str, Any], config: Dict[str, Any]) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "metrics": metrics,
            "config": config,
        },
        path,
    )

def load_checkpoint(path: str, model: nn.Module) -> Dict[str, Any]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint

def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

def compute_binary_metrics(preds, golds):
    n = len(golds)
    if n == 0:
        return {"accuracy": 0.0, "macro_f1": 0.0, "tp": 0, "tn": 0, "fp": 0, "fn": 0}

    correct = sum(p == g for p, g in zip(preds, golds))
    accuracy = correct / n

    tp = sum(p and g for p, g in zip(preds, golds))
    tn = sum((not p) and (not g) for p, g in zip(preds, golds))
    fp = sum(p and (not g) for p, g in zip(preds, golds))
    fn = sum((not p) and g for p, g in zip(preds, golds))

    prec_a = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_a = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_a = 2 * prec_a * rec_a / (prec_a + rec_a) if (prec_a + rec_a) > 0 else 0.0

    prec_b = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_b = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_b = 2 * prec_b * rec_b / (prec_b + rec_b) if (prec_b + rec_b) > 0 else 0.0

    return {
        "accuracy": accuracy,
        "macro_f1": (f1_a + f1_b) / 2,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

def get_aspect_info(aspects_to_use):
    if aspects_to_use == "all":
        return ["theme", "action", "outcome"], 3
    elif aspects_to_use == "theme":
        return ["theme"], 1
    elif aspects_to_use == "action":
        return ["action"], 1
    elif aspects_to_use == "outcome":
        return ["outcome"], 1
    elif aspects_to_use == "theme_action":
        return ["theme", "action"], 2
    elif aspects_to_use == "theme_outcome":
        return ["theme", "outcome"], 2
    elif aspects_to_use == "action_outcome":
        return ["action", "outcome"], 2
    else:
        return ["theme", "action", "outcome"], 3

ASPECT_NAMES, ASPECT_MULTIPLIER = get_aspect_info(ASPECTS_TO_USE)
print(f"Using aspects: {ASPECTS_TO_USE} -> {ASPECT_NAMES}")
print(f"Fusion type: {FUSION_TYPE}")

class PreExtractedAspectLoader:
    def __init__(self, cache_path: str):
        self.cache_path = cache_path
        self.aspects_cache = {}
        self.lookup_hits = 0
        self.lookup_misses = 0
        self._load_cache()
        print(f"✓ PreExtractedAspectLoader initialized with {len(self.aspects_cache)} entries")

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path, "r", encoding="utf-8") as f:
                self.aspects_cache = json.load(f)
            print(f"  Loaded {len(self.aspects_cache)} pre-extracted aspect entries")
        else:
            print(f"  WARNING: Cache file not found at {self.cache_path}")
            print("  Will use empty aspects for all stories")

    def _normalize_text(self, text: str) -> str:
        return " ".join((text or "").split())

    def extract_aspects(self, text: str, story_id: Optional[str] = None) -> Dict[str, str]:
        norm_text = self._normalize_text(text)
        if norm_text in self.aspects_cache:
            self.lookup_hits += 1
            cached = self.aspects_cache[norm_text]
            return {
                "course_of_action": cached.get("coa", ""),
                "outcomes": cached.get("outcomes", ""),
                "abstract_theme": cached.get("theme", ""),
            }

        self.lookup_misses += 1
        return {
            "course_of_action": "",
            "outcomes": "",
            "abstract_theme": "",
        }

    def get_stats(self) -> Dict[str, Any]:
        total = self.lookup_hits + self.lookup_misses
        return {
            "hits": self.lookup_hits,
            "misses": self.lookup_misses,
            "total": total,
            "hit_rate": (self.lookup_hits / total) if total > 0 else 0.0,
        }

class AspectAwareEncoderWithFusion(nn.Module):
    def __init__(self, base_model, hidden_dim=None, aspect_dim=128,
                 aspect_names=None, fusion_type="attention", initial_aspect_weight=0.30):
        super().__init__()
        self.base_model = base_model
        self.hidden_dim = hidden_dim or base_model.config.hidden_size
        self.aspect_dim = aspect_dim
        self.aspect_names = aspect_names or ["theme", "action", "outcome"]
        self.fusion_type = fusion_type
        self.num_aspects = len(self.aspect_names)
        self.aspect_embedding_cache = {}

        if self.fusion_type == "attention":
            self.num_heads = min(4, self.num_aspects)
            self.aspect_dim = ((aspect_dim + self.num_heads - 1) // self.num_heads) * self.num_heads
            print(f"  Adjusted aspect_dim to {self.aspect_dim} for divisibility by {self.num_heads}")

        init_weight = torch.tensor(initial_aspect_weight, dtype=torch.float32).clamp(1e-4, 1 - 1e-4)
        self.aspect_weight_raw = nn.Parameter(torch.logit(init_weight))

        self.aspect_encoders = nn.ModuleDict()
        for aspect_name in self.aspect_names:
            self.aspect_encoders[aspect_name] = nn.Sequential(
                nn.Linear(self.hidden_dim, self.aspect_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.aspect_dim, self.aspect_dim)
            )

        self.aspect_text_encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.aspect_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        for param in self.aspect_text_encoder.parameters():
            param.requires_grad = False
        self.aspect_text_encoder.eval()

        if fusion_type == "attention":
            self.aspect_attention = nn.MultiheadAttention(
                embed_dim=self.aspect_dim,
                num_heads=self.num_heads,
                batch_first=True,
                dropout=0.1
            )
            self.aspect_query = nn.Parameter(torch.randn(1, 1, self.aspect_dim))
        elif fusion_type == "gated":
            self.gates = nn.Sequential(
                nn.Linear(self.aspect_dim * self.num_aspects, self.num_aspects),
                nn.Sigmoid()
            )
        elif fusion_type == "concat":
            self.concat_proj = nn.Linear(self.num_aspects * self.aspect_dim, self.aspect_dim)

        self.final_proj = nn.Sequential(
            nn.Linear(self.aspect_dim, self.hidden_dim),
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(0.1)
        )

        self.base_model.config.use_cache = False
        self.aspect_text_encoder.config.use_cache = False

        print(" AspectAwareEncoderWithFusion initialized")
        print(f"  - Aspects: {self.aspect_names}")
        print(f"  - Num aspects: {self.num_aspects}")
        print(f"  - Aspect dim: {self.aspect_dim}")
        print(f"  - Fusion type: {fusion_type}")
        print(f"  - Learnable aspect weight: {self.get_aspect_weight():.3f}")

    def encode_aspect_text(self, text: str) -> torch.Tensor:
        if not text or len(text) < 5:
            return torch.zeros(self.hidden_dim, device=next(self.base_model.parameters()).device)

        cache_key = " ".join(text.split())
        if cache_key in self.aspect_embedding_cache:
            return self.aspect_embedding_cache[cache_key].to(next(self.base_model.parameters()).device)

        inputs = self.aspect_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(next(self.base_model.parameters()).device)

        with torch.no_grad():
            outputs = self.aspect_text_encoder(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)

        self.aspect_embedding_cache[cache_key] = emb.detach().cpu()
        return emb

    def _get_aspect_text(self, aspects: Dict[str, str], aspect_name: str) -> str:
        if aspect_name == "theme":
            return aspects.get("abstract_theme", "")
        elif aspect_name == "action":
            return aspects.get("course_of_action", "")
        elif aspect_name == "outcome":
            return aspects.get("outcomes", "")
        return ""

    def _encode_single_story_aspects(self, aspects: Dict[str, str]) -> torch.Tensor:
        aspect_embeddings = []
        device = next(self.base_model.parameters()).device

        for aspect_name in self.aspect_names:
            aspect_text = self._get_aspect_text(aspects, aspect_name)
            if aspect_text:
                text_emb = self.encode_aspect_text(aspect_text)
                aspect_emb = self.aspect_encoders[aspect_name](text_emb.unsqueeze(0)).squeeze(0)
            else:
                aspect_emb = torch.zeros(self.aspect_dim, device=device)
            aspect_embeddings.append(aspect_emb)

        return torch.stack(aspect_embeddings)

    def fuse_aspects(self, aspect_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size, num_aspects, _ = aspect_embeddings.shape

        if self.fusion_type == "attention":
            query = self.aspect_query.expand(batch_size, -1, -1)
            attended, attn_weights = self.aspect_attention(query, aspect_embeddings, aspect_embeddings)
            fused = attended.squeeze(1)
            if not self.training:
                self.last_attn_weights = attn_weights.detach().cpu()
        elif self.fusion_type == "gated":
            flat = aspect_embeddings.reshape(batch_size, -1)
            gate_weights = self.gates(flat).unsqueeze(-1)
            fused = (aspect_embeddings * gate_weights).sum(dim=1)
        else:
            fused = aspect_embeddings.reshape(batch_size, -1)
            fused = self.concat_proj(fused)

        fused = self.final_proj(fused)
        fused = F.normalize(fused, p=2, dim=1)
        return fused

    def forward(self, input_ids, attention_mask, aspects: Optional[List[Dict[str, str]]] = None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        base_emb = mean_pool(outputs, attention_mask)

        if aspects is None:
            return base_emb

        if isinstance(aspects, list):
            all_aspect_embs = [self._encode_single_story_aspects(asp) for asp in aspects]
            aspect_embeddings = torch.stack(all_aspect_embs)
            aspect_contribution = self.fuse_aspects(aspect_embeddings)
        else:
            aspect_embs = self._encode_single_story_aspects(aspects)
            aspect_contribution = self.fuse_aspects(aspect_embs.unsqueeze(0))

        aspect_weight = torch.sigmoid(self.aspect_weight_raw)
        final_emb = base_emb + aspect_weight * aspect_contribution
        final_emb = F.normalize(final_emb, p=2, dim=1)
        return final_emb

    def get_aspect_weight(self):
        return torch.sigmoid(self.aspect_weight_raw).item()

    def get_attention_weights(self):
        if hasattr(self, "last_attn_weights"):
            return self.last_attn_weights
        return None

# ========== MODIFIED: SyntheticTripletDataset with prep_text ==========
class SyntheticTripletDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                anchor = prep_text(obj.get("anchor_text", ""))
                ta = prep_text(obj.get("text_a", ""))
                tb = prep_text(obj.get("text_b", ""))
                closer = obj.get("text_a_is_closer")
                if not anchor or not ta or not tb:
                    continue
                if len(anchor) < 20 or len(ta) < 20 or len(tb) < 20:
                    continue
                pos, neg = (ta, tb) if closer else (tb, ta)
                self.data.append((anchor, pos, neg))
        print(f"SyntheticTripletDataset: {len(self.data)} triplets")

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

# ========== MODIFIED: AspectAwareTrackATrainDataset with prep_text ==========
class AspectAwareTrackATrainDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                anchor = prep_text(obj["anchor_text"])
                text_a = prep_text(obj["text_a"])
                text_b = prep_text(obj["text_b"])
                self.data.append({
                    "anchor": anchor,
                    "text_a": text_a,
                    "text_b": text_b,
                    "label": bool(obj.get("text_a_is_closer", False)),
                    "anchor_id": f"anchor_{idx}",
                    "a_id": f"a_{idx}",
                    "b_id": f"b_{idx}",
                })
        print(f"AspectAwareTrackATrainDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "label": torch.tensor(item["label"], dtype=torch.float32),
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

# ========== MODIFIED: AspectAwareTrackATestDataset with prep_text ==========
class AspectAwareTrackATestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.data = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                anchor = prep_text(obj["anchor_text"])
                text_a = prep_text(obj["text_a"])
                text_b = prep_text(obj["text_b"])
                self.data.append({
                    "anchor": anchor,
                    "text_a": text_a,
                    "text_b": text_b,
                    "anchor_id": f"test_anchor_{idx}",
                    "a_id": f"test_a_{idx}",
                    "b_id": f"test_b_{idx}",
                })
        print(f"AspectAwareTrackATestDataset: {len(self.data)} rows")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor_text": item["anchor"],
            "a_text": item["text_a"],
            "b_text": item["text_b"],
            "anchor_aspects": self.aspect_loader.extract_aspects(item["anchor"], item["anchor_id"]),
            "a_aspects": self.aspect_loader.extract_aspects(item["text_a"], item["a_id"]),
            "b_aspects": self.aspect_loader.extract_aspects(item["text_b"], item["b_id"]),
        }

# ========== MODIFIED: AspectAwareTrackBTestDataset with prep_text ==========
class AspectAwareTrackBTestDataset(Dataset):
    def __init__(self, path, aspect_loader):
        self.aspect_loader = aspect_loader
        self.texts = []
        self.ids = []
        with open(path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                obj = json.loads(line)
                self.texts.append(prep_text(obj["text"]))
                self.ids.append(f"test_b_{idx}")
        print(f"AspectAwareTrackBTestDataset: {len(self.texts)} texts")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        return {
            "text": text,
            "aspects": self.aspect_loader.extract_aspects(text, self.ids[idx]),
            "id": self.ids[idx],
        }

def tokenize_texts(tokenizer, texts, max_len):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

def aspect_aware_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "label": torch.stack([b["label"] for b in batch]),
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_test_collate(batch, tokenizer, max_len):
    anchor_enc = tokenize_texts(tokenizer, [b["anchor_text"] for b in batch], max_len)
    a_enc = tokenize_texts(tokenizer, [b["a_text"] for b in batch], max_len)
    b_enc = tokenize_texts(tokenizer, [b["b_text"] for b in batch], max_len)

    return {
        "anchor_input_ids": anchor_enc["input_ids"],
        "anchor_attention_mask": anchor_enc["attention_mask"],
        "a_input_ids": a_enc["input_ids"],
        "a_attention_mask": a_enc["attention_mask"],
        "b_input_ids": b_enc["input_ids"],
        "b_attention_mask": b_enc["attention_mask"],
        "anchor_aspects": [b["anchor_aspects"] for b in batch],
        "a_aspects": [b["a_aspects"] for b in batch],
        "b_aspects": [b["b_aspects"] for b in batch],
    }

def aspect_aware_b_collate(batch, tokenizer, max_len):
    enc = tokenize_texts(tokenizer, [b["text"] for b in batch], max_len)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "aspects": [b["aspects"] for b in batch],
        "id": [b["id"] for b in batch],
    }

def stratified_split_from_labels(dataset, labels, val_ratio=VAL_RATIO, seed=SEED):
    """Return Subset train and validation datasets based on a list of labels."""
    indices = np.arange(len(labels))
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    train_idx, val_idx = next(splitter.split(indices, labels))
    train_ds = Subset(dataset, train_idx.tolist())
    val_ds = Subset(dataset, val_idx.tolist())
    print(f"Dataset split (stratified): train={len(train_ds)} val={len(val_ds)} (val_ratio={val_ratio:.2f})")
    return train_ds, val_ds

def evaluate_aspect_aware_loader(model, loader, criterion):
    model.eval()
    losses, preds, golds = [], [], []

    with torch.no_grad():
        for batch in loader:
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            losses.append(criterion(score, labels).item())

            preds.extend((score > 0).cpu().tolist())
            golds.extend(labels.bool().cpu().tolist())

    metrics = compute_binary_metrics(preds, golds)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    model.train()
    return metrics

def train_stage1_triplet(model, tokenizer, dataset, epochs=1):
    if len(dataset) == 0:
        print("No synthetic data — skipping Stage 1")
        return model

    loader = DataLoader(
        dataset,
        batch_size=STAGE1_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()

    for ep in range(epochs):
        loop = tqdm(loader, desc=f"Stage1 Epoch {ep+1}")
        running = 0.0

        for anchors, positives, negatives in loop:
            texts = list(anchors) + list(positives) + list(negatives)
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(DEVICE)

            emb = model(enc["input_ids"], enc["attention_mask"], aspects=None)

            B = len(anchors)
            a, p, n = emb[:B], emb[B:2*B], emb[2*B:]
            loss = F.triplet_margin_loss(a, p, n, margin=0.5)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running += loss.item()
            loop.set_postfix(loss=running / (loop.n + 1))

    model.eval()
    print("Stage 1 finished.")
    return model

def train_stage2_rank_aspect_aware(model, tokenizer, train_dataset, val_dataset=None, epochs=2,
                                   run_dir=None, experiment_config=None):
    train_loader = DataLoader(
        train_dataset,
        batch_size=STAGE2_BS,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=PRED_BS,
            shuffle=False,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

    param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
    for encoder in model.aspect_encoders.values():
        param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
    param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
    param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})

    if hasattr(model, "aspect_attention"):
        param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
    if hasattr(model, "gates"):
        param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
    if hasattr(model, "concat_proj"):
        param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

    optimizer = torch.optim.AdamW(param_groups)
    criterion = nn.BCEWithLogitsLoss()
    model.train()

    weight_history = []
    epoch_history = []
    best_val_metrics = None
    best_checkpoint_path = None
    best_epoch = None

    if run_dir:
        ensure_dir(run_dir)
        best_checkpoint_path = os.path.join(run_dir, "best_model.pt")

    for ep in range(epochs):
        loop = tqdm(train_loader, desc=f"Stage2 Aspect Epoch {ep+1}")
        running_loss = 0.0

        for batch_idx, batch in enumerate(loop):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            score = sim_a - sim_b
            loss = criterion(score, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=running_loss / (batch_idx + 1))

        current_weight = model.get_aspect_weight()
        weight_history.append(current_weight)
        print(f"  Epoch {ep+1} - Learnable aspect weight: {current_weight:.4f}")

        epoch_record = {
            "epoch": ep + 1,
            "train_loss": running_loss / max(len(train_loader), 1),
            "aspect_weight": current_weight,
        }

        if val_loader is not None:
            val_metrics = evaluate_aspect_aware_loader(model, val_loader, criterion)
            epoch_record["val"] = val_metrics
            print(
                f"  Epoch {ep+1} - Val loss: {val_metrics['loss']:.4f}  "
                f"Val acc: {val_metrics['accuracy']*100:.2f}%  "
                f"Val macro-F1: {val_metrics['macro_f1']*100:.2f}%"
            )
            if best_val_metrics is None or val_metrics["macro_f1"] > best_val_metrics["macro_f1"] or (
                val_metrics["macro_f1"] == best_val_metrics["macro_f1"] and val_metrics["loss"] < best_val_metrics["loss"]
            ):
                best_val_metrics = dict(val_metrics)
                best_epoch = ep + 1
                if best_checkpoint_path:
                    save_checkpoint(best_checkpoint_path, model, best_epoch, best_val_metrics, experiment_config or {})
                    print(f"  Saved new best checkpoint to {best_checkpoint_path}")

        epoch_history.append(epoch_record)

    if best_checkpoint_path and os.path.exists(best_checkpoint_path):
        checkpoint = load_checkpoint(best_checkpoint_path, model)
        best_epoch = checkpoint["epoch"]
        best_val_metrics = checkpoint["metrics"]
        print(
            f"Reloaded best aspect-aware checkpoint from epoch {best_epoch} "
            f"(val macro-F1={best_val_metrics['macro_f1']*100:.2f}%)"
        )

    if run_dir:
        save_json(
            os.path.join(run_dir, "training_history.json"),
            {
                "mode": "aspect_aware",
                "epochs": epoch_history,
                "best_epoch": best_epoch,
                "best_val_metrics": best_val_metrics,
                "weight_history": weight_history,
            },
        )

    model.eval()
    print(f"Stage 2 aspect-aware training finished. Final aspect weight: {model.get_aspect_weight():.4f}")
    print(f"Weight history: {[f'{w:.4f}' for w in weight_history]}")
    return model, best_epoch

def predict_aspect_aware_trackA(model, tokenizer, test_path, out_path, aspect_loader):
    ds = AspectAwareTrackATestDataset(test_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=PRED_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_test_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    preds = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict Track A with aspects"):
            anchor_ids = batch["anchor_input_ids"].to(DEVICE)
            anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
            a_ids = batch["a_input_ids"].to(DEVICE)
            a_mask = batch["a_attention_mask"].to(DEVICE)
            b_ids = batch["b_input_ids"].to(DEVICE)
            b_mask = batch["b_attention_mask"].to(DEVICE)

            anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
            a_emb = model(a_ids, a_mask, batch["a_aspects"])
            b_emb = model(b_ids, b_mask, batch["b_aspects"])

            sim_a = F.cosine_similarity(anchor_emb, a_emb)
            sim_b = F.cosine_similarity(anchor_emb, b_emb)
            preds.extend((sim_a > sim_b).cpu().tolist())

    with open(test_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line, p in zip(fin, preds):
            obj = json.loads(line)
            obj["text_a_is_closer"] = bool(p)
            fout.write(json.dumps(obj) + "\n")

    print("Track A predictions saved:", out_path)
    return preds

def build_trackB_embeddings_aspect_aware(model, tokenizer, in_path, out_npy, aspect_loader):
    ds = AspectAwareTrackBTestDataset(in_path, aspect_loader)
    loader = DataLoader(
        ds,
        batch_size=ENC_BS,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        worker_init_fn=_worker_init_fn,
        generator=dl_generator,
        collate_fn=partial(aspect_aware_b_collate, tokenizer=tokenizer, max_len=MAX_LEN)
    )

    embs = []
    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Encode Track B with aspects"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            emb = model(input_ids, attention_mask, batch["aspects"])
            embs.append(emb.cpu().numpy().astype(np.float32))

    X = np.vstack(embs)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-12, None)
    np.save(out_npy, X)
    print(f"Track B embeddings saved: {out_npy} shape={X.shape}")
    return X

def evaluate_trackA(predictions_path, gold_labels_path):
    preds, golds = [], []
    with open(predictions_path, encoding="utf-8") as fp, open(gold_labels_path, encoding="utf-8") as fg:
        for line_p, line_g in zip(fp, fg):
            preds.append(bool(json.loads(line_p)["text_a_is_closer"]))
            golds.append(bool(json.loads(line_g)["text_a_is_closer"]))

    metrics = compute_binary_metrics(preds, golds)
    metrics["n"] = len(golds)
    metrics["correct"] = sum(p == g for p, g in zip(preds, golds))

    print("\n" + "="*55)
    print("  TRACK A — TEST SET EVALUATION")
    print("="*55)
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def evaluate_trackB(embeddings_npy, gold_labels_path, test_track_b_path):
    X = np.load(embeddings_npy)
    text2idx = {}
    with open(test_track_b_path, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            # Note: we also apply prep_text here for consistency, though not strictly needed for lookup
            text = json.loads(line)["text"].strip()
            text2idx[text] = idx

    correct = 0
    total = 0
    missing = 0
    golds, decisions = [], []

    with open(gold_labels_path, encoding="utf-8") as fg:
        for line in fg:
            obj = json.loads(line)
            anchor = obj["anchor_text"].strip()
            ta = obj["text_a"].strip()
            tb = obj["text_b"].strip()
            gold = bool(obj["text_a_is_closer"])

            idx_a = text2idx.get(anchor)
            idx_ta = text2idx.get(ta)
            idx_tb = text2idx.get(tb)

            if idx_a is None or idx_ta is None or idx_tb is None:
                missing += 1
                continue

            emb_a = X[idx_a]
            emb_ta = X[idx_ta]
            emb_tb = X[idx_tb]

            pred = float(np.dot(emb_a, emb_ta)) > float(np.dot(emb_a, emb_tb))
            golds.append(gold)
            decisions.append(pred)
            if pred == gold:
                correct += 1
            total += 1

    metrics = compute_binary_metrics(decisions, golds)
    metrics["n"] = total
    metrics["missing"] = missing
    metrics["correct"] = correct
    metrics["embedding_dim"] = int(X.shape[1])

    print("\n" + "="*55)
    print("  TRACK B — TEST SET EVALUATION")
    print("="*55)
    print(f"  Embedding dim      : {metrics['embedding_dim']}")
    print(f"  Total triples      : {metrics['n']}")
    print(f"  Skipped (no embed) : {metrics['missing']}")
    print(f"  Correct            : {metrics['correct']}")
    print(f"  Accuracy           : {metrics['accuracy']*100:.2f}%")
    print(f"  Macro F1           : {metrics['macro_f1']*100:.2f}%")
    print("="*55 + "\n")

    return metrics

def print_summary(res_a, res_b):
    print("\n" + "="*55)
    print("  FINAL SUMMARY")
    print("="*55)
    print(f"  Track A accuracy  : {res_a['accuracy']*100:.2f}%")
    print(f"  Track A macro-F1  : {res_a['macro_f1']*100:.2f}%")
    print(f"  Track B accuracy  : {res_b['accuracy']*100:.2f}%")
    print(f"  Track B macro-F1  : {res_b['macro_f1']*100:.2f}%")
    print("-"*55)
    print("  SemEval baselines (Narrative Team CodaBench results):")
    print("    Track A : 64.25%")
    print("    Track B : 69.25%")
    print("="*55 + "\n")

def sanity_checks(test_a, test_b, out_a, out_b):
    n_ta = sum(1 for _ in open(test_a, encoding="utf-8"))
    n_oa = sum(1 for _ in open(out_a, encoding="utf-8"))
    assert n_ta == n_oa, f"Track A line mismatch: {n_ta} vs {n_oa}"

    n_tb = sum(1 for _ in open(test_b, encoding="utf-8"))
    X = np.load(out_b)
    assert X.shape[0] == n_tb, f"Track B row mismatch: {n_tb} vs {X.shape[0]}"
    assert np.isfinite(X).all(), "Track B has NaN/Inf"
    print(f"Sanity OK — TrackA lines={n_oa}  TrackB rows={X.shape[0]} dim={X.shape[1]}")

if __name__ == "__main__":
    print("="*60)
    print("NARRATIVE SIMILARITY WITH PRE-EXTRACTED ASPECTS & LEARNABLE RESIDUAL")
    print("="*60)

    set_seed(SEED)
    configure_determinism()
    ensure_dir(RUNS_DIR)

    run_name = make_run_name()
    run_dir = os.path.join(RUNS_DIR, run_name)
    ensure_dir(run_dir)

    experiment_config = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "seed": SEED,
        "device": DEVICE,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "use_synthetic_stage1": USE_SYNTHETIC_STAGE1,
        "use_aspects": USE_ASPECTS,
        "use_extra_synth_stage2": USE_EXTRA_SYNTH_STAGE2,
        "aspects_to_use": ASPECTS_TO_USE,
        "aspect_names": ASPECT_NAMES,
        "fusion_type": FUSION_TYPE,
        "final_retrain_on_full_dev": FINAL_RETRAIN_ON_FULL_DEV,
        "extra_synth_path": EXTRA_SYNTH_PATH if USE_EXTRA_SYNTH_STAGE2 else None,
    }
    save_json(os.path.join(run_dir, "config.json"), experiment_config)

    print(f"SEED = {SEED}")
    print(f"RUN_NAME = {run_name}")
    print(f"ASPECTS_TO_USE = '{ASPECTS_TO_USE}'")
    print(f"FUSION_TYPE = '{FUSION_TYPE}'")
    print(f"FINAL_RETRAIN_ON_FULL_DEV = {FINAL_RETRAIN_ON_FULL_DEV}")
    print(f"USE_EXTRA_SYNTH_STAGE2 = {USE_EXTRA_SYNTH_STAGE2}")
    print(f"Active aspects: {ASPECT_NAMES}")
    print(f"Aspects cache: {ASPECTS_CACHE_PATH}")

    print("\nRunning ASPECT-AWARE version with pre-extracted aspects")
    aspect_loader = PreExtractedAspectLoader(ASPECTS_CACHE_PATH)

    model = AspectAwareEncoderWithFusion(
        base_model,
        aspect_names=ASPECT_NAMES,
        fusion_type=FUSION_TYPE
    ).to(DEVICE)

    if DEVICE == "cuda":
        model.base_model.gradient_checkpointing_enable()

    # Stage 1: triplet training (synthetic data) – only original, no extra dataset
    if USE_SYNTHETIC_STAGE1:
        synth_ds = SyntheticTripletDataset(SYNTH_PATH)
        model = train_stage1_triplet(model, tokenizer, synth_ds, epochs=1)

    # Save state after Stage 1 for potential full‑dev retraining
    post_stage1_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ------------------------------------------------------------------
    # Stage 2: build combined dataset from dev + extra synthetic data
    # ------------------------------------------------------------------
    dev_dataset = AspectAwareTrackATrainDataset(DEV_TRACK_A_PATH, aspect_loader)
    print(f"Loaded dev data: {len(dev_dataset)} examples")

    extra_dataset = None
    if USE_EXTRA_SYNTH_STAGE2:
        extra_dataset = AspectAwareTrackATrainDataset(EXTRA_SYNTH_PATH, aspect_loader)
        print(f"Loaded extra synthetic data: {len(extra_dataset)} examples")

    if USE_EXTRA_SYNTH_STAGE2 and extra_dataset is not None:
        combined_dataset = ConcatDataset([dev_dataset, extra_dataset])
        # Build label list for stratified split (access underlying .data to avoid double aspect extraction)
        labels = [item["label"] for item in dev_dataset.data] + [item["label"] for item in extra_dataset.data]
        print(f"Combined dataset size: {len(combined_dataset)} examples")
    else:
        combined_dataset = dev_dataset
        labels = [item["label"] for item in dev_dataset.data]
        print(f"Using only dev data: {len(combined_dataset)} examples")

    # Stratified split on combined_dataset
    train_ds, val_ds = stratified_split_from_labels(combined_dataset, labels, val_ratio=VAL_RATIO, seed=SEED)

    model, best_epoch = train_stage2_rank_aspect_aware(
        model, tokenizer, train_ds, val_ds, epochs=4,
        run_dir=run_dir, experiment_config=experiment_config
    )

    print(f"\n✓ Selected aspect weight after dev-split training: {model.get_aspect_weight():.4f}")
    print(f"Aspect cache stats after dev-stage access: {aspect_loader.get_stats()}")

    # ------------------------------------------------------------------
    # Optional full‑dev retraining (on all dev + extra, no validation split)
    # ------------------------------------------------------------------
    if FINAL_RETRAIN_ON_FULL_DEV:
        print("\nRetraining Stage 2 on the full dev+extra set from the post-Stage-1 checkpoint...")
        model.load_state_dict(post_stage1_state)
        model = model.to(DEVICE)
        model.train()

        # free memory
        del train_ds, val_ds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Use the combined_dataset we built earlier, but without validation split
        full_train_ds = combined_dataset   # already includes dev + extra
        epochs_for_full_retrain = best_epoch if best_epoch is not None else 2
        full_dev_bs = 2

        full_loader = DataLoader(
            full_train_ds,
            batch_size=full_dev_bs,
            shuffle=True,
            num_workers=DL_NUM_WORKERS,
            worker_init_fn=_worker_init_fn,
            generator=dl_generator,
            collate_fn=partial(aspect_aware_collate, tokenizer=tokenizer, max_len=MAX_LEN),
        )

        param_groups = [{"params": model.base_model.parameters(), "lr": 3e-6}]
        for encoder in model.aspect_encoders.values():
            param_groups.append({"params": encoder.parameters(), "lr": 1e-5})
        param_groups.append({"params": model.final_proj.parameters(), "lr": 1e-5})
        param_groups.append({"params": [model.aspect_weight_raw], "lr": 1e-3})
        if hasattr(model, "aspect_attention"):
            param_groups.append({"params": model.aspect_attention.parameters(), "lr": 1e-5})
            param_groups.append({"params": [model.aspect_query], "lr": 1e-5})
        if hasattr(model, "gates"):
            param_groups.append({"params": model.gates.parameters(), "lr": 1e-5})
        if hasattr(model, "concat_proj"):
            param_groups.append({"params": model.concat_proj.parameters(), "lr": 1e-5})

        optimizer = torch.optim.AdamW(param_groups, foreach=False)
        criterion = nn.BCEWithLogitsLoss()

        for ep in range(epochs_for_full_retrain):
            loop = tqdm(full_loader, desc=f"Stage2 Full-Dev Epoch {ep+1}")
            running_loss = 0.0
            for batch_idx, batch in enumerate(loop):
                anchor_ids = batch["anchor_input_ids"].to(DEVICE)
                anchor_mask = batch["anchor_attention_mask"].to(DEVICE)
                a_ids = batch["a_input_ids"].to(DEVICE)
                a_mask = batch["a_attention_mask"].to(DEVICE)
                b_ids = batch["b_input_ids"].to(DEVICE)
                b_mask = batch["b_attention_mask"].to(DEVICE)
                labels = batch["label"].to(DEVICE)

                anchor_emb = model(anchor_ids, anchor_mask, batch["anchor_aspects"])
                a_emb = model(a_ids, a_mask, batch["a_aspects"])
                b_emb = model(b_ids, b_mask, batch["b_aspects"])

                score = F.cosine_similarity(anchor_emb, a_emb) - F.cosine_similarity(anchor_emb, b_emb)
                loss = criterion(score, labels)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                running_loss += loss.item()
                loop.set_postfix(loss=running_loss / (batch_idx + 1))

        model.eval()
        print(f"✓ Final aspect weight after full-dev retrain: {model.get_aspect_weight():.4f}")

    # ------------------------------------------------------------------
    # Prediction and evaluation
    # ------------------------------------------------------------------
    predict_aspect_aware_trackA(model, tokenizer, TEST_TRACK_A_PATH, OUT_TRACK_A, aspect_loader)
    build_trackB_embeddings_aspect_aware(model, tokenizer, TEST_TRACK_B_PATH, OUT_TRACK_B, aspect_loader)

    print(f"Aspect cache stats after test-time access: {aspect_loader.get_stats()}")

    sanity_checks(TEST_TRACK_A_PATH, TEST_TRACK_B_PATH, OUT_TRACK_A, OUT_TRACK_B)

    res_a = evaluate_trackA(OUT_TRACK_A, TEST_TRACK_A_LABELS)
    res_b = evaluate_trackB(OUT_TRACK_B, TEST_TRACK_B_LABELS, TEST_TRACK_B_PATH)
    print_summary(res_a, res_b)

    run_summary = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "run_name": run_name,
        "run_dir": run_dir,
        "config": experiment_config,
        "aspect_cache_stats": aspect_loader.get_stats(),
        "final_aspect_weight": model.get_aspect_weight(),
        "track_a": res_a,
        "track_b": res_b,
        "artifacts": {
            "track_a_predictions": OUT_TRACK_A,
            "track_b_embeddings": OUT_TRACK_B,
            "config_path": os.path.join(run_dir, "config.json"),
            "history_path": os.path.join(run_dir, "training_history.json"),
            "best_checkpoint_path": os.path.join(run_dir, "best_model.pt"),
        },
    }
    save_json(os.path.join(run_dir, "summary.json"), run_summary)
    append_jsonl(RESULTS_LOG_PATH, run_summary)

    print(f"Run summary saved: {os.path.join(run_dir, 'summary.json')}")
    print(f"Results log updated: {RESULTS_LOG_PATH}")